# 1. UNIMODAL CHARACTERIZATION — what each modality can do on its own, and where it fails — AeroSync-Mamba v28

**Purpose.** Characterize audio and IMU **each on its own**, and find where each fails. This is **diagnosis, not a
modality contest**. Nothing new is added: no fusion, gating, LACA, HTT time embedding, alignment or consistency loss, or
any other mechanism. v27's fused pipeline is kept unchanged. It is used only as the Stage-0 reproduction gate and as a
reference.

**Shared setup** (read from v24/v27):
- CORRECTED_ALIGNMENT windows: 95 missions, 2.56 s / 1.28 s, Ta = 64, Ts = 128.
- v24's d_model, dropout, optimizer (AdamW 1e-4, weight decay 1e-4, cosine to 1e-6, grad clip 1.0), 20 epochs, batch 32,
  and best-validation-macro-F1 checkpointing.
- Real mamba_ssm, 2 blocks, d_state 16, d_conv 2; attention pooling; the same classifier head.
- Class weights from each fold's training windows. Modality embedding only; v27's day-out folds and inner validation.

**The unimodal loss is class-weighted cross-entropy ONLY.** The InfoNCE alignment loss and the KL consistency loss both
compare two modalities, so they do not exist for a single-modality model.

**Arms.** The pipelines are independent: a dataset returns only its own modality, and a model has no parameters for the
other one.
- **A1 — audio, de-interleaved.** Each 1536-sample buffer is 256 frames × 6 slots. Slot 6 is asserted all-zero and
  dropped. The remaining 5 channels are mean-removed, get a 256-point rFFT, and bins 1–16 are kept, with sqrt, giving 80
  features.
- **A2 — audio, current front-end.** Mono FFT, strongest-64 bins from each fold's training missions, sqrt: 64 features.
- **S1 — IMU, full.** 6 raw channels, fold-train z-score, 1D-CNN encoder.
- **S2 — IMU, vibration only.** Each window's per-channel mean is removed, then divided by the fold-train std of those
  mean-removed values.

**Runs** (seed-major, resumable, each logged to `unimodal_characterization_results.csv` as it finishes):
- Stage 0: fused D gate on the standard split, seed 42 (must give 0.9014 / 0.9080).
- Stage A: day-out, 4 arms × 5 folds × seeds 42/123 = 40 runs.
- Stage B: standard split, 4 arms × seeds 42/123/2024 = 12 runs.
- Stage C (off by default, `RUN_MATCHED_RANDOM`): matched-random, 40 runs.

**Per-run output:** metrics, per-class P/R/F1, the confusion matrix, and a per-window `.npz` holding window/mission ids,
day, class and probabilities.

## 2. Configuration constants & environment
Every fixed value lives in the first cell below. The second installs `mamba_ssm` and reports which fusion backbone is active.

In [ ]:
import copy
import csv
import gc
import glob
import hashlib
import importlib.util
import inspect
import json
import math
import os
import random
import re
import shutil
import subprocess
import sys
import textwrap
import time
import urllib.request
from collections import defaultdict
from datetime import datetime
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

# ==============================================================================
# FIXED CONFIGURATION — every finalized choice (see the summary table in v22)
# ==============================================================================
# --- v28: UNIMODAL CHARACTERIZATION — each modality ON ITS OWN (diagnosis). No fusion, gating, LACA, HTT time
# embedding, alignment/consistency loss or any new mechanism. v27's fused pipeline is kept unchanged as the reference
# (it is trained once, for the Stage-0 reproduction gate). ---
SEEDS = [42, 123]                        # day-out (Stage A) and matched-random (Stage C) seeds
SEEDS_STANDARD = [42, 123, 2024]         # Stage B: v24's original split
CONDITION = "D"                          # FUSED reference (gate only): v22 Condition D, v27's pipeline unchanged
HTT_TIME_EMBED_ENABLED = False
LACA_LAG_BIAS_ENABLED = False
ALIGNMENTS = ["OLD_ALIGNMENT", "CORRECTED_ALIGNMENT"]   # both built as in v24/v27 (CORRECTED depends on OLD); only CORRECTED is used
SPLIT_SEED = 42                          # v24's mission-level split (gate + Stage B)
# Recording days and folds — v27's definitions, unchanged.
DAYS = ["2021-12-22", "2021-12-31", "2022-01-04", "2022-01-05", "2022-01-10"]
DAY_MERGE = {"2021-12-29": "2021-12-31"}
SMALL_FOLD_DAY = "2021-12-22"            # 10 missions: flagged as small in every table
PROTOCOLS = ["day_out", "matched_random"]
PARTITION_SEED = 2026                    # matched-random partition (v27)
INNER_VAL_FRACTION = 0.15                # per fold: ~15% of the fold's training missions per class (min 1) -> checkpoint selection
INNER_VAL_SEED = 2027
RUN_MATCHED_RANDOM = False               # Stage C (40 extra runs) runs ONLY if this is True

# --- the four unimodal arms: (modality, description, input feature dim) ---
ARMS = ["A1", "A2", "S1", "S2"]
ARM_INFO = {
    "A1": ("audio", "AUDIO-DEINTERLEAVED: 5 channels x rFFT bins 1-16 (256-pt), sqrt", 80),
    "A2": ("audio", "AUDIO-CURRENT: mono-FFT strongest-64 (fold-train bins), sqrt", 64),
    "S1": ("sensor", "IMU-FULL: 6 raw channels, fold-train z-score", 6),
    "S2": ("sensor", "IMU-VIBRATION-ONLY: per-window mean removed, fold-train std", 6),
}
A1_BUFFER_LEN = 1536                     # samples per AudioBuffer record (v27 printed N = 1536)
A1_FRAMES, A1_SLOTS, A1_KEEP_SLOTS = 256, 6, 5   # 256 frames x 6 interleaved slots; slot 6 must be all zero -> dropped
A1_BINS = list(range(1, 17))             # per-channel 256-point rFFT (129 bins): keep bins 1..16 (fixed, NOT data-dependent)

# --- analysis (thresholds fixed now; do not tune after seeing results) ---
BOOTSTRAP_RESAMPLES = 2000               # mission-level cluster bootstrap (missions resampled within class; never windows)
BOOTSTRAP_SEED = 2028
CARD_RECALL = 0.70                       # STRONG if recall CI low >= 0.70; WEAK if CI high < 0.70; else UNCERTAIN
N_CONFUSED_PAIRS = 2                     # "where audio fails": the 2 class pairs A1 confuses most
TOP_CELLS = 10
SIGN_CONSISTENT_MIN_DAYS = 4             # a feature's class difference has the same sign on >= 4 of 5 days
IMU_BANDS_HZ = [(2, 10), (10, 30), (30, 90)]
# Fused references. v27 pooled day-out (Condition D, seeds 42/123) — used only if neither v27_test_probs/ nor
# cross_day_audit_results.csv is attached. v24 fused standard-split D macro-F1 for seeds 42/123/2024/7/99.
V27_FUSED_DAYOUT = {"macro_f1": 0.8190, "recall": [0.9906, 0.8291, 0.6561, 0.7151, 0.9093]}
V24_FUSED_STANDARD_F1 = {42: 0.9014, 123: 0.8846, 2024: 0.8966, 7: 0.9157, 99: 0.8846}

# --- Dataset / split ---
DATASET_REPO_URL = "https://github.com/tiiuae/UAV-Realistic-Fault-Dataset.git"
WORK_DIR = Path("/kaggle/working")
DATASET_BASE = WORK_DIR / "tii_uav_dataset"
DATASET_ROOT = DATASET_BASE / "Dataset"
TRAIN_FRACTION = 0.70
VAL_FRACTION = 0.15
N_CLASSES = 5                            # propeller-fault severity 0..4

# --- Windowing ---
WINDOW_SEC = 2.56
STEP_SEC = 1.28
MIN_SENSOR_SAMPLES_PER_WINDOW = 10

# --- Audio: strongest-64 bins (train-only), sqrt(|rFFT|), no window, no normalization ---
N_AUDIO_BINS = 64                        # frequency axis
TA_AUDIO = 64                            # audio time positions per window

# --- Sensor: raw 6 channels (accel x/y/z, gyro x/y/z), no derived features ---
N_SENSOR_CHANNELS = 6
TS_SENSOR = 128                          # sensor time positions per window
SENSOR_STD_EPS = 1e-6

# --- Model ---
D_MODEL = 256
TIME_EMBED_FREQUENCIES = 64
LACA_NUM_HEADS = 4
LACA_BETA = 1.0                          # beta if LACA_LAG_BIAS_ENABLED were True (it is not: Condition D -> 0.0)
LAG_BIAS_HIDDEN = 32                     # mlp g(Δτ): Linear(1,32) -> Tanh -> Linear(32,1) = 97 params (built, unused)
LAG_BIAS_PARAMS = 97
LAG_BIAS_LN_EPS = 1e-5
MAMBA_NUM_BLOCKS = 2
MAMBA_D_STATE = 16
MAMBA_D_CONV = 2
MAMBA_EXPAND = 2
EXPECTED_MAMBA_PARAMS = 874_496          # 2 x 437,248 (real mamba_ssm)
DROPOUT = 0.15

# --- Loss: L_cls (class-weighted CE) + 0.1 * L_align (InfoNCE) + 0.1 * L_cons (KL) ---
LAMBDA_ALIGN = 0.1
LAMBDA_CONS = 0.1
INFONCE_TEMPERATURE = 0.07

# --- Optimization ---
EPOCHS = 20
BATCH_SIZE = 32
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
LR_MIN = 1e-6
GRAD_CLIP_NORM = 1.0
NUM_WORKERS = 2

# --- v23: corrected temporal anchor (CORRECTED_ALIGNMENT only — OLD_ALIGNMENT reads none of this) ---
# audio_start_offset_s = (recording_started_time - sensor_t0) / 1e6, both raw microsecond timestamps of the same clock
AUDIO_EXCLUDE_RATIO = 0.5                # exclude a mission if audio_start_offset_s / mission_duration_s >= this
EXPECTED_EXCLUDED_MISSIONS = {"rosbag2_2021_12_29-09_10_31", "rosbag2_2021_12_31-10_12_31",
                              "rosbag2_2021_12_31-09_58_13", "rosbag2_2021_12_31-10_04_22"}
EXPECTED_N_MISSIONS = {"OLD_ALIGNMENT": 99, "CORRECTED_ALIGNMENT": 95}
KNOWN_AUDIO_OFFSETS_S = {"rosbag2_2021_12_22-09_14_31": 24.38}   # earlier investigation (motor-start event at t = 24.5 s)
KNOWN_OFFSET_TOLERANCE_S = 0.05
EXPECTED_MEDIAN_OFFSET_S = 19.2          # median over all missions, earlier investigation
MEDIAN_OFFSET_TOLERANCE_S = 1.0
N_OFFSETS_TO_PRINT = 8
# Where CORRECTED windows start: "audio_onset" = the 2.56 s / 1.28 s grid starts at the mission's audio onset, so every
# window lies where BOTH modalities were recorded (v18 already ends the grid at min(sensor end, audio end)).
# "sensor_t0" = v18's grid start (t = 0); windows before the onset then get all-zero audio (np.interp left=0.0).
CORRECTED_WINDOW_START = "audio_onset"
# Strongest-64 bins for CORRECTED: "own_train" = recomputed (same method) from CORRECTED's own training missions;
# "old_bins" = reuse OLD_ALIGNMENT's frozen bins.
CORRECTED_BIN_SOURCE = "own_train"
assert CORRECTED_WINDOW_START in ("audio_onset", "sensor_t0") and CORRECTED_BIN_SOURCE in ("own_train", "old_bins")

# --- Outputs (the results CSV is NEVER renamed or overwritten: the notebook resumes from it) ---
CHECKPOINT_DIR = WORK_DIR / "v28_checkpoints"
PROBS_DIR = WORK_DIR / "v28_test_probs"  # per run: {arm}_{protocol}_{fold}_{seed}.npz (window/mission ids, day, class, probs)
RESULTS_CSV = WORK_DIR / "unimodal_characterization_results.csv"
RESUME_CSV_GLOBS = ["/kaggle/input/**/unimodal_characterization_results.csv"]   # an earlier session's output, if attached
RESUME_NPZ_GLOBS = ["/kaggle/input/**/v28_test_probs/*.npz"]
V27_PROBS_GLOB = "/kaggle/input/**/v27_test_probs/{name}"     # fused day-out reference (optional)
V27_CSV_GLOB = "/kaggle/input/**/cross_day_audit_results.csv"
for _d in (CHECKPOINT_DIR, PROBS_DIR):
    _d.mkdir(parents=True, exist_ok=True)

# --- Reference numbers ---
# v22 Condition D (HTT-time OFF, LACA-lag OFF), seed 42 — Artifacts/Version 22/temporal_2x2_ablation_results.csv.
# Same data, architecture and seed as OLD_ALIGNMENT here: THIS is what OLD_ALIGNMENT must reproduce.
V22_D_SEED42 = {"accuracy": 0.9097345132743363, "macro_f1": 0.90640272771588, "balanced_accuracy": 0.9040236181473512,
                "macro_auroc": 0.9890678208140686, "class3_recall": 0.8859649122807017, "class4_recall": 0.8706467661691543,
                "confusion_matrix": [[284, 0, 0, 0, 2], [3, 197, 4, 0, 1], [0, 1, 170, 38, 1], [0, 0, 10, 202, 16], [0, 0, 3, 23, 175]],
                "best_epoch": 17, "init_sha1": "5850841e699f", "rng_sha1": "5641b401720d"}
# v18 beta=0.0, seed 42 = v22 Condition A (HTT time embedding ON). A DIFFERENT architecture from Condition D:
# printed for context only, NOT a reproduction target.
V18_BETA0_SEED42 = {"accuracy": 0.9088495575221239, "macro_f1": 0.9057330017972403, "balanced_accuracy": 0.9030033658152373,
                    "macro_auroc": 0.9891867537737307, "class3_recall": 0.8947368421052632, "class4_recall": 0.8507462686567164}
SAME_SEED_TOLERANCE = 0.0005             # same-seed reproducibility (code correctness), not a finding threshold
SEED_TO_SEED_STD_F1 = 0.0160             # v15/v16 measured seed-to-seed std of test macro-F1 = the noise floor here
REFERENCE_SEED42_RNG_PREFIX = "5641b401720d"   # torch CPU RNG right after seed-42 model construction (v10-v22)
REFERENCE_DATA = {
    "bin_indices": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 245, 246, 247, 248, 249, 250, 251, 252, 253, 254, 255,
                    258, 259, 260, 261, 262, 263, 264, 265, 266, 267, 501, 502, 503, 504, 505, 506, 507, 508,
                    509, 510, 511, 514, 515, 516, 517, 518, 519, 520, 521, 522, 523, 754, 757, 758, 759, 760,
                    761, 762, 763, 764, 765, 766],
    "sensor_fingerprint": {
        "train": (4934, -6167105.225, 808427.456, (906, 1020, 1004, 1008, 996)),
        "val":   (954, -1193206.652, 156301.664, (126, 229, 199, 190, 210)),
        "test":  (1130, -1413563.944, 185127.438, (286, 205, 210, 228, 201)),
    },
}
# sha1[:16] of the source of every v18 data function / method (textwrap.dedent + rstrip). The two renamed functions are
# hashed under their v18 names. OLD_ALIGNMENT's code is checked against these: identical hash = identical code.
V18_DATA_SOURCE_SHA1 = {
    "parse_mission": "48132d5e84609fc4",
    "split_missions": "e60744ab660df986",
    "select_strongest_bins": "fb7d15f798514139",
    "audio_features": "d957dda051fb003d",
    "slice_windows": "0d493fd15803e0de",
    "UAVWindowDataset.__init__": "4a7708aec1baca55",
    "UAVWindowDataset.__len__": "15a9aa16ef54ea0a",
    "UAVWindowDataset.__getitem__": "5b6e2898c53b5989",
    "split_fingerprint": "195d93e4771675ca",
}

# --- v27 references ---
# Initial weights / torch RNG right after construction, per seed (v18 CSV) — every run of a seed starts here.
SEED_INIT_REFERENCE = {42: ("5850841e699f", "5641b401720d"), 123: ("74be0883c62b", "0a4b3be44b18"),
                       2024: ("92852955eef1", "619c0e0a6b4d"), 7: ("7f5d0cd2e7cb", "9fdcf777d285"),
                       99: ("e40c78169f57", "81dcd4638a88")}
# CORRECTED_ALIGNMENT as v23/v24 built it (v24 run output) + v23's CORRECTED function sources (sha1 of the source).
V23_CORRECTED_DATA = {"n_missions": 95, "missions": {"train": 67, "val": 13, "test": 15},
                      "windows": {"train": 3737, "val": 686, "test": 815},
                      "class_counts": {"train": (701, 765, 706, 796, 769), "test": (214, 156, 108, 179, 158)}}
V23_CORRECTED_SOURCE_SHA1 = {"parse_mission_CORRECTED_ALIGNMENT": "49d4b2f5a1359025", "slice_windows_CORRECTED_ALIGNMENT": "8cf47da958eda742"}
# Reproduction gate (b): v24 CORRECTED_ALIGNMENT Condition D seed 42 (Artifacts/Version 24/alignment_5seed_paired_results.csv).
V24_CORRECTED_D_SEED42 = {"accuracy": 0.9079754601226994, "macro_f1": 0.9014428634096197, "balanced_accuracy": 0.8939724876561621,
                          "macro_auroc": 0.992763216249706, "class3_recall": 0.9553072625698324, "class4_recall": 0.7468354430379747,
                          "confusion_matrix": [[212, 2, 0, 0, 0], [0, 153, 2, 1, 0], [0, 2, 86, 19, 1], [0, 0, 3, 171, 5], [0, 1, 0, 39, 118]],
                          "best_epoch": 15, "best_val_macro_f1": 0.927039, "train_seconds": 153.4}
GATE_TOLERANCE = 0.0005                  # macro-F1 AND accuracy must both be within this of v24

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")


def set_all_seeds(seed: int) -> None:
    """Seed python `random`, numpy, torch CPU and every CUDA device; deterministic cuDNN."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


_n_runs = {"A": len(ARMS) * len(DAYS) * len(SEEDS), "B": len(ARMS) * len(SEEDS_STANDARD),
           "C": len(ARMS) * len(DAYS) * len(SEEDS) if RUN_MATCHED_RANDOM else 0}
print(f"UNIMODAL CHARACTERIZATION: arms {ARMS} | Stage 0 gate (fused D, seed 42) + Stage A day-out {_n_runs['A']} + Stage B "
      f"standard {_n_runs['B']} + Stage C matched-random {_n_runs['C']} ({'ON' if RUN_MATCHED_RANDOM else 'OFF'}) = "
      f"{1 + sum(_n_runs.values())} runs | results -> {RESULTS_CSV} (resumable)")
for _a in ARMS:
    print(f"    {_a}: {ARM_INFO[_a][1]} | input dim {ARM_INFO[_a][2]}")
print("Unimodal loss = class-weighted CE ONLY (InfoNCE alignment and KL consistency need two modalities). Modality "
      "embedding only; no time embedding; no LACA; no auxiliary heads.")
print(f"Shared (v24): d_model {D_MODEL}, dropout {DROPOUT}, Mamba {MAMBA_NUM_BLOCKS} x (d_state {MAMBA_D_STATE}, d_conv "
      f"{MAMBA_D_CONV}, expand {MAMBA_EXPAND}), attention pooling, classifier head | {EPOCHS} epochs, batch {BATCH_SIZE}, "
      f"AdamW {LEARNING_RATE} (wd {WEIGHT_DECAY}), cosine to {LR_MIN}, grad clip {GRAD_CLIP_NORM}, best epoch by validation macro-F1")

In [ ]:
# ==============================================================================
# mamba_ssm installation + fusion-backbone resolution.
# Order: an already-installed package, else a prebuilt wheel matched to this
# Python / torch / CUDA / C++ ABI (the reliable path on Kaggle), else a
# time-capped source build. If none succeeds, or there is no CUDA GPU, the
# pure-PyTorch fallback block is used and announced below.
# ==============================================================================
PIP_TIMEOUT_S = 240


def _env_tags():
    py_tag = f"cp{sys.version_info.major}{sys.version_info.minor}"
    torch_mm = ".".join(torch.__version__.split("+")[0].split(".")[:2])
    try:
        cxx11_abi = bool(torch._C._GLIBCXX_USE_CXX11_ABI)
    except AttributeError:
        cxx11_abi = False
    return py_tag, torch_mm, torch.version.cuda, cxx11_abi


def _prebuilt_wheel_url(repo, py_tag, torch_mm, cuda_ver, cxx11_abi):
    """Best-matching .whl asset in `repo`'s GitHub releases (exact CUDA version first, then CUDA major)."""
    req = urllib.request.Request(f"https://api.github.com/repos/{repo}/releases",
                                 headers={"User-Agent": "kaggle-notebook", "Accept": "application/vnd.github+json"})
    try:
        with urllib.request.urlopen(req, timeout=20) as resp:
            releases = json.load(resp)
    except Exception as exc:
        print(f"    could not query github.com/{repo}/releases: {exc!r}")
        return None
    abi = f"cxx11abi{'TRUE' if cxx11_abi else 'FALSE'}"
    cuda_exact = f"cu{cuda_ver.replace('.', '')}"
    cuda_major = f"cu{cuda_ver.split('.')[0]}"
    exact, major_only = [], []
    for rel in releases:
        for asset in rel.get("assets", []):
            name = asset.get("name", "")
            if not (name.endswith(".whl") and py_tag in name and abi in name and f"torch{torch_mm}" in name):
                continue
            if cuda_exact in name:
                exact.append(asset["browser_download_url"])
            elif cuda_major in name:
                major_only.append(asset["browser_download_url"])
    chosen = exact or major_only
    return chosen[0] if chosen else None


def _pip_install(args):
    """pip with a hard timeout and captured output, so a stuck build can never hang the kernel."""
    try:
        proc = subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + args,
                              timeout=PIP_TIMEOUT_S, capture_output=True, text=True, check=False)
    except subprocess.TimeoutExpired:
        print(f"    pip install {args} exceeded {PIP_TIMEOUT_S}s — killed")
        return False
    if proc.returncode != 0:
        print(f"    pip install {args} failed (exit {proc.returncode}): {(proc.stderr or proc.stdout)[-400:]}")
    return proc.returncode == 0


def install_mamba_ssm():
    if importlib.util.find_spec("mamba_ssm") is not None:
        return "already installed"
    tags = _env_tags()
    print(f"Installing mamba_ssm + causal-conv1d (python={tags[0]}, torch={tags[1]}, cuda={tags[2]}, cxx11abi={tags[3]})")
    urls = [_prebuilt_wheel_url(repo, *tags) for repo in ("Dao-AILab/causal-conv1d", "state-spaces/mamba")]
    if all(urls) and all(_pip_install([u, "--no-deps"]) for u in urls):
        return "prebuilt wheel"
    print("  no complete prebuilt-wheel match — trying a time-capped source build")
    if _pip_install(["causal-conv1d>=1.2.0", "--no-build-isolation"]) and _pip_install(["mamba-ssm", "--no-build-isolation"]):
        return "source build"
    return "failed"


MAMBA_CLASS = None
if torch.cuda.is_available():
    print(f"mamba_ssm install: {install_mamba_ssm()}")
    try:
        from mamba_ssm import Mamba as MAMBA_CLASS
    except ImportError as exc:
        print(f"mamba_ssm import failed: {exc!r}")

if MAMBA_CLASS is not None:
    FUSION_BACKBONE = "mamba_ssm.Mamba (real selective-scan SSM, CUDA)"
    print(f"✅ FUSION BACKBONE: {FUSION_BACKBONE}")
else:
    FUSION_BACKBONE = "FALLBACK pure-PyTorch gated-conv block (NOT a state-space model)"
    print("🚨" * 45)
    print(f"🚨  FUSION BACKBONE: {FUSION_BACKBONE}")
    print("🚨  " + ("no CUDA GPU detected" if not torch.cuda.is_available() else "mamba_ssm could not be installed/imported")
          + " — results will NOT match the reference numbers, which were produced with real mamba_ssm.")
    print("🚨" * 45)

## 3a. Data — OLD_ALIGNMENT (v18's preprocessing, unchanged)
This is v18's data cell. Two functions are renamed and nothing else changes: `parse_mission` becomes
`parse_mission_OLD_ALIGNMENT`, and `slice_windows` becomes `slice_windows_OLD_ALIGNMENT`. All 99 missions are used, with
audio anchored at t = 0. At the end of the cell, every data function's source is hashed and compared with v18's, and
the bins and per-split fingerprints are compared with v18's reference values. Either a code change or a data change
would show up.

**v24 addition (bookkeeping only).** For each OLD window, the build loop now records its start time. It recomputes
the start with the same grid and skip rule that `slice_windows_OLD_ALIGNMENT` uses, and asserts that the recomputed
starts agree with every window's real timestamps. The time-matched control needs these starts. No function is changed,
and the v18 code and data checks below still apply.

**v27:** unchanged. OLD_ALIGNMENT is built only because the CORRECTED cell reuses its split, parser summary and bins.

In [ ]:
# ==============================================================================
# DATA — OLD_ALIGNMENT: v18's preprocessing, UNCHANGED (audio anchored at t = 0, all 99 missions).
# parse missions -> mission-level split -> strongest-64 bins from TRAINING audio only (frozen) -> 2.56 s windows
# with real per-window timestamps -> datasets (sensor standardized with TRAINING stats).
# v23: parse_mission -> parse_mission_OLD_ALIGNMENT and slice_windows -> slice_windows_OLD_ALIGNMENT (renamed only).
# ==============================================================================
if not DATASET_ROOT.exists():
    print(f"Cloning {DATASET_REPO_URL} ...")
    subprocess.run(["git", "clone", "--depth", "1", DATASET_REPO_URL, str(DATASET_BASE)], check=True)


def parse_mission_OLD_ALIGNMENT(mission_dir: Path):
    """Real 6-axis IMU (seconds from mission start) + raw audio chunks with per-chunk timestamps.
    Audio chunk duration = the mission's IMU-measured duration / number of chunks (both come from the same
    continuous recording; the audio log carries no per-chunk timestamp)."""
    sensor_files = list(mission_dir.glob("*SensorCombined*.jsonl"))
    audio_files = list(mission_dir.glob("*AudioBuffer*.jsonl"))
    if not sensor_files or not audio_files:
        return None

    sensor_times, accel, gyro = [], [], []
    with open(sensor_files[0], "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            try:
                rec = json.loads(line)
                t_sec = float(rec["timestamp"]) / 1e6
                sensor_times.append(t_sec)
                accel.append(rec.get("accelerometer_m_s2", [0.0, 0.0, 9.81]))
                gyro.append(rec.get("gyro_rad", [0.0, 0.0, 0.0]))
            except Exception:
                continue
    if len(sensor_times) < 50:
        return None
    sensor_times = np.array(sensor_times, dtype=np.float64)
    sensor_times = sensor_times - sensor_times[0]
    telemetry = np.concatenate([np.array(accel, dtype=np.float32), np.array(gyro, dtype=np.float32)], axis=-1)

    audio_chunks = []
    with open(audio_files[0], "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            try:
                rec = json.loads(line)
                chunk = rec.get("audio_data", rec.get("buffer", rec.get("data", [])))
                if len(chunk) > 0:
                    audio_chunks.append(chunk)
            except Exception:
                continue
    if len(audio_chunks) < 10:
        return None
    audio_chunks = np.array(audio_chunks, dtype=np.float32)

    chunk_duration_s = (sensor_times[-1] - sensor_times[0]) / len(audio_chunks)
    audio_times = np.arange(len(audio_chunks), dtype=np.float64) * chunk_duration_s
    return {"sensor_times": sensor_times, "telemetry": telemetry, "audio_times": audio_times, "audio_chunks": audio_chunks}


def split_missions(dataset_root: Path):
    """Mission-level stratified split (70 / 15 / 15 per class) with its own fixed seed."""
    class_missions = defaultdict(list)
    for class_dir in sorted(dataset_root.iterdir()):
        if not class_dir.is_dir() or class_dir.name.startswith("."):
            continue
        try:
            label = int(class_dir.name)
        except ValueError:
            continue
        for mission_dir in sorted(class_dir.iterdir()):
            if mission_dir.is_dir() and not mission_dir.name.startswith("."):
                class_missions[label].append(mission_dir)

    random.seed(SPLIT_SEED)
    splits = {"train": {}, "val": {}, "test": {}}
    for label in sorted(class_missions):
        missions = class_missions[label]
        random.shuffle(missions)
        n_train = max(1, int(len(missions) * TRAIN_FRACTION))
        n_val = max(1, int(len(missions) * VAL_FRACTION))
        splits["train"][label] = missions[:n_train]
        splits["val"][label] = missions[n_train:n_train + n_val]
        splits["test"][label] = missions[n_train + n_val:]
        print(f"Class {label} ({len(missions)} flights) -> train {n_train}, val {len(splits['val'][label])}, "
              f"test {len(splits['test'][label])}")
    return splits


def select_strongest_bins(train_missions, n_bins):
    """Indices of the n_bins rFFT magnitude bins with the highest mean magnitude over TRAINING audio only
    (unsupervised energy, no labels). Returned sorted; frozen and reused for val/test."""
    magnitude_sum, n_chunks, fft_len = None, 0, None
    for mission in train_missions:
        mag = np.abs(np.fft.rfft(mission["audio_chunks"], axis=1))
        fft_len = mag.shape[1] if fft_len is None else min(fft_len, mag.shape[1])
        mag = mag[:, :fft_len]
        if magnitude_sum is not None and len(magnitude_sum) > fft_len:
            magnitude_sum = magnitude_sum[:fft_len]
        magnitude_sum = mag.sum(axis=0) if magnitude_sum is None else magnitude_sum + mag.sum(axis=0)
        n_chunks += mag.shape[0]
    mean_magnitude = magnitude_sum / n_chunks
    return np.sort(np.argsort(mean_magnitude)[::-1][:n_bins]).astype(int)


def audio_features(audio_chunks, bin_indices):
    """raw chunk -> rFFT (no window) -> |.| -> frozen strongest bins -> sqrt (no normalization)."""
    mag = np.abs(np.fft.rfft(audio_chunks, axis=1))
    valid = bin_indices[bin_indices < mag.shape[1]]
    feats = np.zeros((mag.shape[0], len(bin_indices)), dtype=np.float32)
    feats[:, :len(valid)] = np.sqrt(mag[:, valid]).astype(np.float32)
    return feats


def slice_windows_OLD_ALIGNMENT(mission, label, bin_indices):
    """2.56 s windows (1.28 s step). Sensor: TS_SENSOR real IMU samples chosen evenly by index inside the window;
    audio: TA_AUDIO real chunk timestamps chosen the same way (a window with fewer than 2 audio chunks — the
    recording edge — falls back to an even grid). tau_a / tau_s = those REAL timestamps relative to window start."""
    s_times, telemetry = mission["sensor_times"], mission["telemetry"]
    a_times = mission["audio_times"]
    a_feats = audio_features(mission["audio_chunks"], bin_indices)
    max_time = min(s_times[-1], a_times[-1])

    windows, t_start = [], 0.0
    while t_start + WINDOW_SEC <= max_time:
        t_end = t_start + WINDOW_SEC
        s_mask = (s_times >= t_start) & (s_times <= t_end)
        if s_mask.sum() < MIN_SENSOR_SAMPLES_PER_WINDOW:
            t_start += STEP_SEC
            continue

        s_idx = np.where(s_mask)[0]
        tau_s_abs = s_times[s_idx[np.linspace(0, len(s_idx) - 1, TS_SENSOR).round().astype(int)]]
        x_s = np.zeros((TS_SENSOR, N_SENSOR_CHANNELS), dtype=np.float32)
        for c in range(N_SENSOR_CHANNELS):
            x_s[:, c] = np.interp(tau_s_abs, s_times[s_mask], telemetry[s_mask, c])

        a_mask = (a_times >= t_start) & (a_times <= t_end)
        if a_mask.sum() >= 2:
            a_idx = np.where(a_mask)[0]
            tau_a_abs = a_times[a_idx[np.linspace(0, len(a_idx) - 1, TA_AUDIO).round().astype(int)]]
        else:
            tau_a_abs = np.linspace(t_start, t_end, TA_AUDIO)
        x_a = np.zeros((TA_AUDIO, len(bin_indices)), dtype=np.float32)
        for c in range(len(bin_indices)):
            x_a[:, c] = np.interp(tau_a_abs, a_times, a_feats[:, c], left=0.0, right=0.0)

        windows.append({
            "x_a": x_a,
            "x_s": x_s,
            "label": int(label),
            "tau_a": (tau_a_abs - t_start).astype(np.float32),
            "tau_s": (tau_s_abs - t_start).astype(np.float32),
            "window_sec": np.float32(WINDOW_SEC),
        })
        t_start += STEP_SEC
    return windows


class UAVWindowDataset(Dataset):
    """Windows -> tensors. Sensor channels are standardized with mean/std that must come from the TRAINING
    split (computed here only when none are passed, i.e. for the training set). tau is normalized by the
    window length."""

    def __init__(self, windows, sensor_mean=None, sensor_std=None):
        self.windows = windows
        if sensor_mean is None or sensor_std is None:
            all_s = np.concatenate([w["x_s"] for w in windows], axis=0)
            sensor_mean = all_s.mean(axis=0, keepdims=True)
            sensor_std = all_s.std(axis=0, keepdims=True) + SENSOR_STD_EPS
        self.sensor_mean, self.sensor_std = sensor_mean, sensor_std

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):
        w = self.windows[idx]
        window_sec = float(w["window_sec"])
        return {
            "x_a": torch.tensor(w["x_a"], dtype=torch.float32),
            "x_s": torch.tensor((w["x_s"] - self.sensor_mean) / self.sensor_std, dtype=torch.float32),
            "tau_a": torch.tensor(w["tau_a"], dtype=torch.float32) / window_sec,
            "tau_s": torch.tensor(w["tau_s"], dtype=torch.float32) / window_sec,
            "label": torch.tensor(w["label"], dtype=torch.long),
        }


def split_fingerprint(windows):
    """(n windows, sum of raw sensor values, sum of sensor timestamps, class counts) — deterministic data check."""
    x_s = np.stack([w["x_s"] for w in windows]).astype(np.float64)
    tau_s = np.stack([w["tau_s"] for w in windows]).astype(np.float64)
    counts = tuple(np.bincount([w["label"] for w in windows], minlength=N_CLASSES).tolist())
    return (len(windows), round(float(x_s.sum()), 3), round(float(tau_s.sum()), 3), counts)


# --- 1. split + parse ---
mission_splits = split_missions(DATASET_ROOT)
split_dirs = {s: [m for label in sorted(mission_splits[s]) for m in mission_splits[s][label]] for s in mission_splits}
assert not set(split_dirs["train"]) & (set(split_dirs["val"]) | set(split_dirs["test"])), "LEAKAGE: mission in two splits"
parsed_missions = {m: parse_mission_OLD_ALIGNMENT(m) for s in split_dirs for m in tqdm(split_dirs[s], desc=f"parse {s}", leave=False)}
# v23: per-mission raw-data summary, so the CORRECTED parser can be checked to read the same samples and chunks.
OLD_MISSION_SUMMARY = {m.name: (len(p["sensor_times"]), len(p["audio_chunks"]), float(p["sensor_times"][-1]))
                       for m, p in parsed_missions.items() if p is not None}

# --- 2. strongest-64 bins from TRAINING audio only ---
bin_source_missions = [m for m in split_dirs["train"] if parsed_missions[m] is not None]
assert set(bin_source_missions) <= set(split_dirs["train"])
SELECTED_BINS = select_strongest_bins([parsed_missions[m] for m in bin_source_missions], N_AUDIO_BINS)
assert len(SELECTED_BINS) == N_AUDIO_BINS
print(f"[LEAKAGE CHECK] strongest-{N_AUDIO_BINS} bins computed from {len(bin_source_missions)} TRAINING missions only "
      f"(0 of {len(split_dirs['val']) + len(split_dirs['test'])} val/test missions) -> frozen for all splits: "
      f"{SELECTED_BINS.tolist()}")

# --- 3. windows (the same frozen bins for every split) ---
def _old_window_starts(mission):
    """v24 bookkeeping: the t_start of every window slice_windows_OLD_ALIGNMENT returns, in order (same grid from 0.0,
    same += STEP_SEC accumulation, same end rule, same sensor-count skip). Does not build any window."""
    s_times, a_times = mission["sensor_times"], mission["audio_times"]
    max_time = min(s_times[-1], a_times[-1])
    starts, t_start = [], 0.0
    while t_start + WINDOW_SEC <= max_time:
        if (((s_times >= t_start) & (s_times <= t_start + WINDOW_SEC)).sum()) >= MIN_SENSOR_SAMPLES_PER_WINDOW:
            starts.append(t_start)
        t_start += STEP_SEC
    return starts


def _starts_match_windows(mission, windows, starts):
    """v24 bookkeeping check: rebuild each window's tau_s exactly as slice_windows_OLD_ALIGNMENT does (the samples inside
    [t, t + WINDOW_SEC] in FILE order, evenly chosen by index) from the recomputed start t, and compare with the stored
    tau_s. Valid whether or not the IMU timestamps are sorted."""
    s_times = mission["sensor_times"]
    for w, t in zip(windows, starts):
        s_idx = np.where((s_times >= t) & (s_times <= t + WINDOW_SEC))[0]
        tau_s = (s_times[s_idx[np.linspace(0, len(s_idx) - 1, TS_SENSOR).round().astype(int)]] - t).astype(np.float32)
        if not np.array_equal(tau_s, w["tau_s"]):
            return False
    return True


split_windows, split_window_missions = {}, {}   # v23: the mission of every window is recorded next to it (same order)
split_window_starts = {}                        # v24: ... and its start time (s from the first IMU sample)
NON_MONOTONIC_SENSOR_MISSIONS = {}              # v24 diagnostic: missions whose IMU timestamps go backwards somewhere
for split in ("train", "val", "test"):
    split_windows[split], split_window_missions[split], split_window_starts[split] = [], [], []
    for label in sorted(mission_splits[split]):
        for m in mission_splits[split][label]:
            if parsed_missions[m] is not None:
                _w = slice_windows_OLD_ALIGNMENT(parsed_missions[m], label, SELECTED_BINS)
                split_windows[split].extend(_w)
                split_window_missions[split].extend([m.name] * len(_w))
                _st = _old_window_starts(parsed_missions[m])
                assert len(_st) == len(_w), f"{m.name}: {len(_st)} recomputed starts vs {len(_w)} windows"
                assert _starts_match_windows(parsed_missions[m], _w, _st), \
                    f"{m.name}: recomputed window starts disagree with the windows' own timestamps"
                _back = int((np.diff(parsed_missions[m]["sensor_times"]) < 0).sum())
                if _back:
                    NON_MONOTONIC_SENSOR_MISSIONS[m.name] = _back
                split_window_starts[split].extend(_st)
    print(f"{split:<5}: {len(split_windows[split])} windows | audio {TA_AUDIO}x{N_AUDIO_BINS}, sensor {TS_SENSOR}x{N_SENSOR_CHANNELS}")
DATA_FINGERPRINT = {s: split_fingerprint(split_windows[s]) for s in split_windows}
print(f"[WINDOW STARTS] every OLD window's recomputed start verified by rebuilding its tau_s exactly ✅ | missions with "
      f"non-monotonic IMU timestamps: {len(NON_MONOTONIC_SENSOR_MISSIONS)}"
      + (f" {NON_MONOTONIC_SENSOR_MISSIONS} (backward steps per mission; v18's slicer picks samples in file order, so its "
         f"windows and these starts are unaffected — the rebuild above checks exactly that)" if NON_MONOTONIC_SENSOR_MISSIONS else ""))

# --- 4. datasets: sensor mean/std from the TRAINING split only, reused for val/test ---
train_ds = UAVWindowDataset(split_windows["train"])
val_ds = UAVWindowDataset(split_windows["val"], train_ds.sensor_mean, train_ds.sensor_std)
test_ds = UAVWindowDataset(split_windows["test"], train_ds.sensor_mean, train_ds.sensor_std)
_train_s = np.concatenate([w["x_s"] for w in split_windows["train"]], axis=0)
_val_s = np.concatenate([w["x_s"] for w in split_windows["val"]], axis=0)
assert val_ds.sensor_mean is train_ds.sensor_mean and test_ds.sensor_std is train_ds.sensor_std
assert np.array_equal(train_ds.sensor_mean, _train_s.mean(axis=0, keepdims=True))
print(f"[LEAKAGE CHECK] sensor standardization stats = TRAINING-split stats (same objects reused for val/test) | "
      f"train mean {np.round(train_ds.sensor_mean.ravel(), 3).tolist()} vs val's own mean "
      f"{np.round(_val_s.mean(axis=0), 3).tolist()} (not used)")
del _train_s, _val_s

# --- 5. real timestamps reach the model: tau must differ across windows ---
_tau_a = np.stack([train_ds[i]["tau_a"].numpy() for i in range(len(train_ds))])
_tau_s = np.stack([train_ds[i]["tau_s"].numpy() for i in range(len(train_ds))])
_n_unique_a, _n_unique_s = len(np.unique(_tau_a, axis=0)), len(np.unique(_tau_s, axis=0))
assert _n_unique_a > 1 and _n_unique_s > 1, "real timestamps are NOT reaching the model (tau identical for every window)"
assert not np.allclose(_tau_s, np.linspace(0.0, 1.0, TS_SENSOR)[None, :]), "tau_s equals a synthetic linspace grid"
print(f"[TIMESTAMP CHECK] {len(train_ds)} training windows: {_n_unique_a} distinct tau_a rows, {_n_unique_s} distinct "
      f"tau_s rows -> real per-window timestamps (Condition D does not use them)")
del _tau_a, _tau_s

# --- 6. v23: OLD_ALIGNMENT is v18, byte for byte (code) and value for value (data) ---
def _source_sha1(obj, v18_name=None):
    """sha1[:16] of obj's source (dedented, rstripped); a renamed function is compared under its v18 name."""
    s = textwrap.dedent(inspect.getsource(obj)).rstrip()
    if v18_name is not None:
        s = s.replace(f"def {obj.__name__}(", f"def {v18_name}(", 1)
    return hashlib.sha1(s.encode()).hexdigest()[:16]


_old_code = {"parse_mission": (parse_mission_OLD_ALIGNMENT, "parse_mission"),
             "split_missions": (split_missions, None),
             "select_strongest_bins": (select_strongest_bins, None),
             "audio_features": (audio_features, None),
             "slice_windows": (slice_windows_OLD_ALIGNMENT, "slice_windows"),
             "UAVWindowDataset.__init__": (UAVWindowDataset.__init__, None),
             "UAVWindowDataset.__len__": (UAVWindowDataset.__len__, None),
             "UAVWindowDataset.__getitem__": (UAVWindowDataset.__getitem__, None),
             "split_fingerprint": (split_fingerprint, None)}
assert set(_old_code) == set(V18_DATA_SOURCE_SHA1)
OLD_SOURCE_CHECK = {}
for _name, (_obj, _v18_name) in _old_code.items():
    try:
        _h = _source_sha1(_obj, _v18_name)
        OLD_SOURCE_CHECK[_name] = _h == V18_DATA_SOURCE_SHA1[_name]
        print(f"[OLD == v18 CODE] {_name:<30} sha1 {_h} vs v18 {V18_DATA_SOURCE_SHA1[_name]} -> "
              + ("identical ✅" if OLD_SOURCE_CHECK[_name] else "🚨 DIFFERENT — v18's preprocessing was modified"))
    except (OSError, TypeError) as _e:
        OLD_SOURCE_CHECK[_name] = None
        print(f"[OLD == v18 CODE] {_name:<30} source not readable here ({_e}); the data fingerprint below still applies")
OLD_DATA_CHECK = {"strongest-64 bin indices": SELECTED_BINS.tolist() == REFERENCE_DATA["bin_indices"],
                  **{f"{s} windows / sensor values / timestamps / class counts": DATA_FINGERPRINT[s] ==
                     REFERENCE_DATA["sensor_fingerprint"][s] for s in ("train", "val", "test")}}
for _name, _ok in OLD_DATA_CHECK.items():
    print(f"[OLD == v18 DATA] {_name:<58}: {'match ✅' if _ok else '🚨 MISMATCH'}")
OLD_N_MISSIONS = len(OLD_MISSION_SUMMARY)
print(f"[OLD MISSIONS] {OLD_N_MISSIONS} parsed missions (expected {EXPECTED_N_MISSIONS['OLD_ALIGNMENT']}) -> "
      + ("✅" if OLD_N_MISSIONS == EXPECTED_N_MISSIONS["OLD_ALIGNMENT"] else "🚨 UNEXPECTED COUNT"))
if not all(OLD_DATA_CHECK.values()) or False in OLD_SOURCE_CHECK.values():
    print("\n" + "🚨" * 40 + "\n🚨 OLD_ALIGNMENT does NOT reproduce v18's preprocessing — the OLD vs CORRECTED comparison "
          "is not against the historical pipeline.\n" + "🚨" * 40)
else:
    print("✅ OLD_ALIGNMENT = v18's preprocessing: identical code, identical strongest-64 bins, identical windows / sensor "
          "values / timestamps / class counts in every split. The 99-mission bin selection is v18's, unchanged.")

OLD_ALIGNMENT = {"name": "OLD_ALIGNMENT", "n_missions": OLD_N_MISSIONS, "bins": SELECTED_BINS,
                 "split_windows": split_windows, "window_missions": split_window_missions,
                 "window_starts": {s: np.array(v) for s, v in split_window_starts.items()},
                 "fingerprint": DATA_FINGERPRINT, "train_ds": train_ds, "val_ds": val_ds, "test_ds": test_ds,
                 "split_missions": {s: sorted({n for n in split_window_missions[s]}) for s in split_windows}}
del parsed_missions   # raw audio of 99 missions; the CORRECTED cell re-reads the raw files itself
gc.collect()

## 3b. Data — CORRECTED_ALIGNMENT (new, alongside OLD)
- **Parser.** `parse_mission_CORRECTED_ALIGNMENT` is a copy of the OLD parser with one change: the audio time axis is
  anchored at `recording_started_time`. That value is read from this mission's first AudioBuffer record, never
  hardcoded. `sensor_t0` is the raw timestamp of the first IMU sample, the same one OLD subtracts.
- **Checks.**
  - The first few offsets are printed, and `rosbag2_2021_12_22-09_14_31` must read ≈ 24.38 s.
  - The median offset must be ≈ 19.2 s.
  - `recording_started_time` must be constant within each mission.
  - The sample and chunk counts must equal OLD's.
- **Exclusion.** Missions with `offset / duration ≥ 0.5` are excluded from **every** split of this dataset only. They
  are printed with their ratios and checked against the 4 expected names.
- **Split, bins, windows.** The split is the same as OLD's. Bins are recomputed from CORRECTED's training missions, and
  windows start at the audio onset (see the header).
- **Summary.** Both datasets' sizes are printed side by side at the end.
- **v27 change (bookkeeping only).** The parsed missions are **kept** (`CORRECTED_PARSED`) instead of freed. Each fold
  re-windows them with its own strongest-64 bins. No function changes, and the v23 hash checks still apply.

In [ ]:
# ==============================================================================
# DATA — CORRECTED_ALIGNMENT: the SAME raw files and the SAME mission split, with the audio time axis anchored at the
# real recording start (recording_started_time) instead of t = 0. Missions whose audio starts in the second half of
# the flight are excluded from this dataset only. Nothing in the OLD_ALIGNMENT cell above is modified.
# ==============================================================================
def parse_mission_CORRECTED_ALIGNMENT(mission_dir: Path):
    """parse_mission_OLD_ALIGNMENT with ONE change — the audio time axis:
        OLD:       chunk_duration_s = mission_duration_s / n_chunks
                   audio_times      = arange(n_chunks) * chunk_duration_s
        CORRECTED: audio_start_offset_s = (recording_started_time - sensor_t0) / 1e6
                   chunk_duration_s     = (mission_duration_s - audio_start_offset_s) / n_chunks
                   audio_times          = audio_start_offset_s + arange(n_chunks) * chunk_duration_s
    recording_started_time: the FIRST AudioBuffer record's field (raw microseconds; constant within a mission).
    sensor_t0: raw microsecond timestamp of the first IMU sample (the sample OLD's sensor_times[0] refers to)."""
    sensor_files = list(mission_dir.glob("*SensorCombined*.jsonl"))
    audio_files = list(mission_dir.glob("*AudioBuffer*.jsonl"))
    if not sensor_files or not audio_files:
        return None

    sensor_times, accel, gyro, sensor_t0_us = [], [], [], None
    with open(sensor_files[0], "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            try:
                rec = json.loads(line)
                t_sec = float(rec["timestamp"]) / 1e6
                sensor_times.append(t_sec)
                accel.append(rec.get("accelerometer_m_s2", [0.0, 0.0, 9.81]))
                gyro.append(rec.get("gyro_rad", [0.0, 0.0, 0.0]))
                if sensor_t0_us is None:
                    sensor_t0_us = float(rec["timestamp"])
            except Exception:
                continue
    if len(sensor_times) < 50:
        return None
    sensor_times = np.array(sensor_times, dtype=np.float64)
    sensor_times = sensor_times - sensor_times[0]
    telemetry = np.concatenate([np.array(accel, dtype=np.float32), np.array(gyro, dtype=np.float32)], axis=-1)

    audio_chunks, recording_started_time, rst_values = [], None, set()
    with open(audio_files[0], "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            try:
                rec = json.loads(line)
                if "recording_started_time" in rec:
                    if recording_started_time is None:
                        recording_started_time = float(rec["recording_started_time"])
                    rst_values.add(float(rec["recording_started_time"]))
                chunk = rec.get("audio_data", rec.get("buffer", rec.get("data", [])))
                if len(chunk) > 0:
                    audio_chunks.append(chunk)
            except Exception:
                continue
    if len(audio_chunks) < 10:
        return None
    if recording_started_time is None:
        print(f"🚨 {mission_dir.name}: no recording_started_time in {audio_files[0].name} -> cannot anchor the audio; "
              f"mission left out of CORRECTED_ALIGNMENT")
        return None
    audio_chunks = np.array(audio_chunks, dtype=np.float32)

    mission_duration_s = float(sensor_times[-1] - sensor_times[0])
    audio_start_offset_s = (recording_started_time - sensor_t0_us) / 1e6
    chunk_duration_s = (mission_duration_s - audio_start_offset_s) / len(audio_chunks)
    audio_times = audio_start_offset_s + np.arange(len(audio_chunks), dtype=np.float64) * chunk_duration_s
    return {"sensor_times": sensor_times, "telemetry": telemetry, "audio_times": audio_times, "audio_chunks": audio_chunks,
            "recording_started_time": recording_started_time, "sensor_t0_us": sensor_t0_us,
            "n_distinct_recording_started_time": len(rst_values), "audio_start_offset_s": audio_start_offset_s,
            "mission_duration_s": mission_duration_s, "offset_ratio": audio_start_offset_s / mission_duration_s,
            "chunk_duration_s": chunk_duration_s, "old_chunk_duration_s": mission_duration_s / len(audio_chunks)}


def slice_windows_CORRECTED_ALIGNMENT(mission, label, bin_indices):
    """slice_windows_OLD_ALIGNMENT with ONE change — where the window grid starts:
    CORRECTED_WINDOW_START = "audio_onset": the first window starts at the audio onset (audio_times[0]), so every window
    lies inside the span where BOTH modalities were recorded (the end is min(sensor end, audio end), as in OLD).
    CORRECTED_WINDOW_START = "sensor_t0": OLD's start (t = 0); pre-onset windows then carry all-zero audio."""
    s_times, telemetry = mission["sensor_times"], mission["telemetry"]
    a_times = mission["audio_times"]
    a_feats = audio_features(mission["audio_chunks"], bin_indices)
    max_time = min(s_times[-1], a_times[-1])

    windows = []
    t_start = max(0.0, float(a_times[0])) if CORRECTED_WINDOW_START == "audio_onset" else 0.0
    while t_start + WINDOW_SEC <= max_time:
        t_end = t_start + WINDOW_SEC
        s_mask = (s_times >= t_start) & (s_times <= t_end)
        if s_mask.sum() < MIN_SENSOR_SAMPLES_PER_WINDOW:
            t_start += STEP_SEC
            continue

        s_idx = np.where(s_mask)[0]
        tau_s_abs = s_times[s_idx[np.linspace(0, len(s_idx) - 1, TS_SENSOR).round().astype(int)]]
        x_s = np.zeros((TS_SENSOR, N_SENSOR_CHANNELS), dtype=np.float32)
        for c in range(N_SENSOR_CHANNELS):
            x_s[:, c] = np.interp(tau_s_abs, s_times[s_mask], telemetry[s_mask, c])

        a_mask = (a_times >= t_start) & (a_times <= t_end)
        if a_mask.sum() >= 2:
            a_idx = np.where(a_mask)[0]
            tau_a_abs = a_times[a_idx[np.linspace(0, len(a_idx) - 1, TA_AUDIO).round().astype(int)]]
        else:
            tau_a_abs = np.linspace(t_start, t_end, TA_AUDIO)
        x_a = np.zeros((TA_AUDIO, len(bin_indices)), dtype=np.float32)
        for c in range(len(bin_indices)):
            x_a[:, c] = np.interp(tau_a_abs, a_times, a_feats[:, c], left=0.0, right=0.0)

        windows.append({
            "x_a": x_a,
            "x_s": x_s,
            "label": int(label),
            "tau_a": (tau_a_abs - t_start).astype(np.float32),
            "tau_s": (tau_s_abs - t_start).astype(np.float32),
            "window_sec": np.float32(WINDOW_SEC),
        })
        t_start += STEP_SEC
    return windows


# --- 1. parse every mission of the SAME split (mission_splits / split_dirs from the OLD cell: SPLIT_SEED 42) ---
corrected_parsed = {m: parse_mission_CORRECTED_ALIGNMENT(m) for s in split_dirs
                    for m in tqdm(split_dirs[s], desc=f"parse {s} (corrected)", leave=False)}
mission_home = {m: (s, label) for s in mission_splits for label in mission_splits[s] for m in mission_splits[s][label]}
_ok = {m: p for m, p in corrected_parsed.items() if p is not None}
_same_raw = [m.name for m, p in _ok.items()
             if OLD_MISSION_SUMMARY.get(m.name) != (len(p["sensor_times"]), len(p["audio_chunks"]), float(p["sensor_times"][-1]))]
print(f"[RAW DATA CHECK] {len(_ok)} missions parsed by the CORRECTED parser; IMU samples, audio chunks and duration equal "
      f"OLD's for " + ("every mission ✅" if not _same_raw and len(_ok) == len(OLD_MISSION_SUMMARY) else
                       f"all but {_same_raw} 🚨"))
_non_const = {m.name: p["n_distinct_recording_started_time"] for m, p in _ok.items() if p["n_distinct_recording_started_time"] != 1}
RST_CONSTANT = not _non_const
print("[METADATA] recording_started_time is constant within every mission ✅" if RST_CONSTANT else
      f"🚨 [METADATA] recording_started_time varies within {_non_const} (distinct values per mission) — the first "
      f"record's value is used, as specified")

# --- 2. verification: offsets read from the metadata ---
_by_name = {m.name: (m, p) for m, p in _ok.items()}
print(f"\n[OFFSETS] first {N_OFFSETS_TO_PRINT} missions (by name): audio_start_offset_s = (recording_started_time - "
      f"sensor_t0) / 1e6")
print(f"    {'mission':<30} {'class':>5} {'split':>5} {'sensor_t0 (us)':>18} {'rec_started (us)':>18} {'offset s':>9} "
      f"{'dur s':>8} {'ratio':>6} {'chunk ms OLD':>12} {'chunk ms NEW':>12}")
for _name in sorted(_by_name)[:N_OFFSETS_TO_PRINT] + [n for n in KNOWN_AUDIO_OFFSETS_S if n in _by_name]:
    _m, _p = _by_name[_name]
    _s, _lab = mission_home[_m]
    print(f"    {_name:<30} {_lab:>5} {_s:>5} {_p['sensor_t0_us']:>18.0f} {_p['recording_started_time']:>18.0f} "
          f"{_p['audio_start_offset_s']:>9.3f} {_p['mission_duration_s']:>8.2f} {_p['offset_ratio']:>6.3f} "
          f"{1e3 * _p['old_chunk_duration_s']:>12.3f} {1e3 * _p['chunk_duration_s']:>12.3f}")
KNOWN_OFFSET_CHECK = {}
for _name, _expected in KNOWN_AUDIO_OFFSETS_S.items():
    if _name not in _by_name:
        KNOWN_OFFSET_CHECK[_name] = False
        print(f"🚨 [KNOWN OFFSET] {_name} not found among the parsed missions")
        continue
    _got = _by_name[_name][1]["audio_start_offset_s"]
    KNOWN_OFFSET_CHECK[_name] = abs(_got - _expected) <= KNOWN_OFFSET_TOLERANCE_S
    print(f"[KNOWN OFFSET] {_name}: computed {_got:.3f} s vs earlier investigation ~{_expected:.2f} s -> "
          + ("✅ metadata read correctly" if KNOWN_OFFSET_CHECK[_name] else f"🚨 MISMATCH (> ±{KNOWN_OFFSET_TOLERANCE_S} s)"))
_offsets = np.array([p["audio_start_offset_s"] for p in _ok.values()])
MEDIAN_OFFSET_S = float(np.median(_offsets))
MEDIAN_OFFSET_OK = abs(MEDIAN_OFFSET_S - EXPECTED_MEDIAN_OFFSET_S) <= MEDIAN_OFFSET_TOLERANCE_S
print(f"[OFFSETS] over {len(_offsets)} missions: median {MEDIAN_OFFSET_S:.2f} s (expected ~{EXPECTED_MEDIAN_OFFSET_S}) "
      f"{'✅' if MEDIAN_OFFSET_OK else '🚨'} | min {_offsets.min():.2f} s | max {_offsets.max():.2f} s | "
      f"{int((_offsets < 0).sum())} negative")
print(f"[OLD MISPLACEMENT] under OLD_ALIGNMENT, chunk k sits at k*D/n but was recorded at offset + k*(D - offset)/n: the "
      f"audio is placed offset*(1 - k/n) too EARLY — {MEDIAN_OFFSET_S:.1f} s at the start of a median mission, shrinking "
      f"to 0 at the end (mean {MEDIAN_OFFSET_S / 2:.1f} s = {MEDIAN_OFFSET_S / 2 / WINDOW_SEC:.1f} windows)")

# --- 3. exclusion rule: offset / duration >= AUDIO_EXCLUDE_RATIO -> out of CORRECTED_ALIGNMENT (every split) ---
_ratios = {n: p["offset_ratio"] for n, (m, p) in _by_name.items()}
EXCLUDED_MISSIONS = {n: r for n, r in sorted(_ratios.items()) if r >= AUDIO_EXCLUDE_RATIO}
print(f"\n[EXCLUSION] rule: audio_start_offset_s / mission_duration_s >= {AUDIO_EXCLUDE_RATIO} -> excluded from "
      f"CORRECTED_ALIGNMENT only ({len(EXCLUDED_MISSIONS)} missions):")
for _name, _r in EXCLUDED_MISSIONS.items():
    _m, _p = _by_name[_name]
    _s, _lab = mission_home[_m]
    print(f"    EXCLUDED {_name:<30} ratio {_r:.3f} | offset {_p['audio_start_offset_s']:.2f} s of "
          f"{_p['mission_duration_s']:.2f} s | class {_lab} | split {_s}")
print("    nearest kept missions (highest ratios below the threshold): " + ", ".join(
    f"{n} {r:.3f}" for n, r in sorted(((n, r) for n, r in _ratios.items() if r < AUDIO_EXCLUDE_RATIO), key=lambda t: -t[1])[:3]))
EXCLUSION_MATCHES = set(EXCLUDED_MISSIONS) == EXPECTED_EXCLUDED_MISSIONS
if EXCLUSION_MATCHES:
    print(f"[EXCLUSION] ✅ computed exclusion set == the 4 expected missions {sorted(EXPECTED_EXCLUDED_MISSIONS)}")
else:
    print("\n" + "🚨" * 40)
    print(f"🚨 EXCLUSION SET DOES NOT MATCH THE EXPECTED LIST\n🚨   computed but not expected: "
          f"{sorted(set(EXCLUDED_MISSIONS) - EXPECTED_EXCLUDED_MISSIONS)}\n🚨   expected but not computed: "
          f"{sorted(EXPECTED_EXCLUDED_MISSIONS - set(EXCLUDED_MISSIONS))} (ratios: "
          f"{ {n: round(_ratios[n], 3) for n in EXPECTED_EXCLUDED_MISSIONS if n in _ratios} })\n🚨 The computed set is "
          f"what is applied. Check the metadata before trusting this run.")
    print("🚨" * 40 + "\n")
corrected_splits = {s: {label: [m for m in mission_splits[s][label]
                                if corrected_parsed[m] is not None and m.name not in EXCLUDED_MISSIONS]
                        for label in mission_splits[s]} for s in mission_splits}
CORRECTED_N_MISSIONS = sum(len(v) for s in corrected_splits for v in corrected_splits[s].values())
print(f"[CORRECTED MISSIONS] {CORRECTED_N_MISSIONS} missions (expected {EXPECTED_N_MISSIONS['CORRECTED_ALIGNMENT']}) -> "
      + ("✅" if CORRECTED_N_MISSIONS == EXPECTED_N_MISSIONS["CORRECTED_ALIGNMENT"] else "🚨 UNEXPECTED COUNT"))

# --- 4. strongest-64 bins: from CORRECTED's TRAINING missions only (or OLD's frozen bins) ---
_corr_train = [m for label in sorted(corrected_splits["train"]) for m in corrected_splits["train"][label]]
assert set(_corr_train) <= set(split_dirs["train"])
if CORRECTED_BIN_SOURCE == "own_train":
    CORRECTED_BINS = select_strongest_bins([corrected_parsed[m] for m in _corr_train], N_AUDIO_BINS)
else:
    CORRECTED_BINS = OLD_ALIGNMENT["bins"].copy()
_overlap = len(set(CORRECTED_BINS.tolist()) & set(OLD_ALIGNMENT["bins"].tolist()))
CORRECTED_BINS_SAME = CORRECTED_BINS.tolist() == OLD_ALIGNMENT["bins"].tolist()
print(f"[BINS] CORRECTED strongest-{N_AUDIO_BINS} ({CORRECTED_BIN_SOURCE}: {len(_corr_train)} TRAINING missions, "
      f"0 val/test) -> {_overlap}/{N_AUDIO_BINS} shared with OLD -> "
      + ("identical to OLD's bins ✅" if CORRECTED_BINS_SAME else
         f"DIFFERENT: only in CORRECTED {sorted(set(CORRECTED_BINS.tolist()) - set(OLD_ALIGNMENT['bins'].tolist()))}, "
         f"only in OLD {sorted(set(OLD_ALIGNMENT['bins'].tolist()) - set(CORRECTED_BINS.tolist()))}"))

# --- 5. windows ---
corr_windows, corr_window_missions = {}, {}
for split in ("train", "val", "test"):
    corr_windows[split], corr_window_missions[split] = [], []
    for label in sorted(corrected_splits[split]):
        for m in corrected_splits[split][label]:
            _w = slice_windows_CORRECTED_ALIGNMENT(corrected_parsed[m], label, CORRECTED_BINS)
            corr_windows[split].extend(_w)
            corr_window_missions[split].extend([m.name] * len(_w))
    print(f"{split:<5}: {len(corr_windows[split])} windows | audio {TA_AUDIO}x{N_AUDIO_BINS}, sensor {TS_SENSOR}x{N_SENSOR_CHANNELS}")
CORRECTED_FINGERPRINT = {s: split_fingerprint(corr_windows[s]) for s in corr_windows}

# --- 6. datasets: sensor mean/std from CORRECTED's TRAINING split only ---
corr_train_ds = UAVWindowDataset(corr_windows["train"])
corr_val_ds = UAVWindowDataset(corr_windows["val"], corr_train_ds.sensor_mean, corr_train_ds.sensor_std)
corr_test_ds = UAVWindowDataset(corr_windows["test"], corr_train_ds.sensor_mean, corr_train_ds.sensor_std)
assert corr_val_ds.sensor_mean is corr_train_ds.sensor_mean and corr_test_ds.sensor_std is corr_train_ds.sensor_std
print(f"[LEAKAGE CHECK] CORRECTED sensor standardization = CORRECTED training-split stats | mean "
      f"{np.round(corr_train_ds.sensor_mean.ravel(), 3).tolist()}")

# --- 7. audio coverage: how many windows carry all-zero audio (no chunk inside them) ---
def _zero_audio(windows):
    rows = np.stack([np.abs(w["x_a"]).sum(axis=1) == 0 for w in windows])   # (n windows, TA_AUDIO)
    return int(rows.all(axis=1).sum()), int(rows.any(axis=1).sum())
ZERO_AUDIO = {}
for _tag, _wins in (("OLD_ALIGNMENT", OLD_ALIGNMENT["split_windows"]), ("CORRECTED_ALIGNMENT", corr_windows)):
    _all = [w for s in ("train", "val", "test") for w in _wins[s]]
    ZERO_AUDIO[_tag] = _zero_audio(_all)
    print(f"[AUDIO COVERAGE] {_tag:<20}: {ZERO_AUDIO[_tag][0]} of {len(_all)} windows have ALL-zero audio, "
          f"{ZERO_AUDIO[_tag][1]} have some all-zero audio rows")

CORRECTED_ALIGNMENT = {"name": "CORRECTED_ALIGNMENT", "n_missions": CORRECTED_N_MISSIONS, "bins": CORRECTED_BINS,
                       "split_windows": corr_windows, "window_missions": corr_window_missions,
                       "fingerprint": CORRECTED_FINGERPRINT, "train_ds": corr_train_ds, "val_ds": corr_val_ds,
                       "test_ds": corr_test_ds,
                       "split_missions": {s: sorted({n for n in corr_window_missions[s]}) for s in corr_windows},
                       "excluded": EXCLUDED_MISSIONS,
                       "offsets_s": {n: p["audio_start_offset_s"] for n, (m, p) in _by_name.items()}}
# v27: the parsed missions are KEPT (not deleted): every fold re-windows them with its own strongest-64 bins.
CORRECTED_PARSED = corrected_parsed
del _ok, _by_name
gc.collect()

# --- 8. both datasets, side by side ---
DATASETS = {"OLD_ALIGNMENT": OLD_ALIGNMENT, "CORRECTED_ALIGNMENT": CORRECTED_ALIGNMENT}
print("\n" + "=" * 100 + "\nDATASET SIZES — both built, both kept\n" + "=" * 100)
print(f"    {'':<22}" + "".join(f"{n:>24}" for n in DATASETS))
print(f"    {'missions (total)':<22}" + "".join(f"{d['n_missions']:>24}" for d in DATASETS.values()))
for _s in ("train", "val", "test"):
    print(f"    {'missions ' + _s:<22}" + "".join(f"{len(d['split_missions'][_s]):>24}" for d in DATASETS.values()))
for _s in ("train", "val", "test"):
    print(f"    {'windows ' + _s:<22}" + "".join(f"{len(d['split_windows'][_s]):>24}" for d in DATASETS.values()))
print(f"    {'windows (total)':<22}" + "".join(f"{sum(len(v) for v in d['split_windows'].values()):>24}" for d in DATASETS.values()))
for _s in ("train", "test"):
    print(f"    {'class counts ' + _s:<22}" + "".join(f"{str(d['fingerprint'][_s][3]):>24}" for d in DATASETS.values()))
for _name, _r in EXCLUDED_MISSIONS.items():
    print(f"    excluded from CORRECTED: {_name} (split {mission_home[[m for m in mission_home if m.name == _name][0]][0]}, "
          f"ratio {_r:.3f})")

## 3c. v23 reuse checks
The two CORRECTED functions are hashed and compared with v23's source. Mission, window and class counts are compared
with v23/v24. These feed reproduction gate (a).

In [ ]:
# ==============================================================================
# v23 REUSE CHECKS (CORRECTED code + data == v23) (v27: v24's class diagnostic and controls removed)
# ==============================================================================
CORRECTED_SOURCE_CHECK = {}
for _name, _obj in (("parse_mission_CORRECTED_ALIGNMENT", parse_mission_CORRECTED_ALIGNMENT),
                    ("slice_windows_CORRECTED_ALIGNMENT", slice_windows_CORRECTED_ALIGNMENT)):
    try:
        _h = _source_sha1(_obj)
        CORRECTED_SOURCE_CHECK[_name] = _h == V23_CORRECTED_SOURCE_SHA1[_name]
        print(f"[CORRECTED == v23 CODE] {_name:<36} sha1 {_h} vs v23 {V23_CORRECTED_SOURCE_SHA1[_name]} -> "
              + ("identical ✅" if CORRECTED_SOURCE_CHECK[_name] else "🚨 DIFFERENT"))
    except (OSError, TypeError) as _e:
        CORRECTED_SOURCE_CHECK[_name] = None
        print(f"[CORRECTED == v23 CODE] {_name:<36} source not readable here ({_e}); the counts below still apply")
CORRECTED_DATA_CHECK = {
    "missions": CORRECTED_ALIGNMENT["n_missions"] == V23_CORRECTED_DATA["n_missions"],
    **{f"{s} missions": len(CORRECTED_ALIGNMENT["split_missions"][s]) == V23_CORRECTED_DATA["missions"][s] for s in ("train", "val", "test")},
    **{f"{s} windows": len(CORRECTED_ALIGNMENT["split_windows"][s]) == V23_CORRECTED_DATA["windows"][s] for s in ("train", "val", "test")},
    **{f"{s} class counts": tuple(CORRECTED_ALIGNMENT["fingerprint"][s][3]) == V23_CORRECTED_DATA["class_counts"][s] for s in ("train", "test")},
    "exclusion set == the 4 expected": EXCLUSION_MATCHES,
    "bins identical to OLD (as in v23)": CORRECTED_BINS_SAME,
}
for _name, _ok in CORRECTED_DATA_CHECK.items():
    print(f"[CORRECTED == v23 DATA] {_name:<36}: {'match ✅' if _ok else '🚨 MISMATCH'}")
DATA_REUSE_OK = (all(OLD_DATA_CHECK.values()) and False not in OLD_SOURCE_CHECK.values()
                 and all(CORRECTED_DATA_CHECK.values()) and False not in CORRECTED_SOURCE_CHECK.values())
print(("✅ Both preprocessing paths are reused exactly as built in v23 (OLD == v18, CORRECTED == v23): same code, same "
       "missions, same windows.") if DATA_REUSE_OK else
      "🚨 A preprocessing path differs from v23 — seed 42 will NOT be reused, and results are not comparable to v23.")

## 4. Model definition — v22 Condition D
This is the v22 model cell, unchanged apart from the probe at the end. `AeroSyncMamba(htt_time=False, laca_lag=False)`
is Condition D:
- HTT returns `norm(z + e_m)`.
- LACA adds `0.0 · LN_grid(g(Δτ))`, which is exactly 0.

Both temporal modules are still built. That keeps the initial weights and the RNG stream identical to v18/v22 seed 42
(`5850841e699f` / `5641b401720d`). Both runs, OLD and CORRECTED, start from these same weights.

**v28:** unchanged from v27; in v28 it is used only for the Stage-0 fused reproduction gate.

In [ ]:
# ==============================================================================
# MODEL — encoders -> HTT (modality + continuous-time embedding of the REAL tau)
# -> bi-directional LACA (lag-aware cross-attention; lag bias beta * LN_grid(g(Δτ)), beta = 1.0 when
#    LACA_LAG_BIAS_ENABLED else 0.0; 4 heads). v22: HTT's time embedding is switchable too.
# v23: only Condition D is built (HTT-time OFF, LACA-lag OFF) — the classes below are v22's, unchanged.
# -> [H_a ; SEP ; H_s] -> 2 Mamba blocks -> attention pooling -> classifier.
# Auxiliary per-modality heads feed only the consistency loss.
# ==============================================================================
class ContinuousTimeEmbedding(nn.Module):
    def __init__(self, d_model: int, num_frequencies: int = TIME_EMBED_FREQUENCIES):
        super().__init__()
        self.frequencies = nn.Parameter(
            torch.exp(torch.linspace(0, math.log(1000.0), num_frequencies)), requires_grad=False
        )
        self.mlp = nn.Sequential(
            nn.Linear(2 * num_frequencies, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model),
        )

    def forward(self, tau: torch.Tensor) -> torch.Tensor:
        if tau.dim() == 2:
            tau = tau.unsqueeze(-1)
        angles = tau * self.frequencies * 2.0 * math.pi
        return self.mlp(torch.cat([torch.sin(angles), torch.cos(angles)], dim=-1))


class HeterogeneousTemporalTokenization(nn.Module):
    """token = LayerNorm(features + modality embedding [+ time embedding(tau)]); modality 0 = audio, 1 = sensor.
    v22: time_enabled = HTT_TIME_EMBED_ENABLED. When False the time embedding is skipped entirely (no temporal
    position information enters through HTT); the module is still built so every condition draws the same RNG."""

    def __init__(self, d_model: int, time_enabled: bool = True):
        super().__init__()
        self.d_model = d_model
        self.time_enabled = time_enabled
        self.modality_embeddings = nn.Embedding(2, d_model)
        self.time_embedding = ContinuousTimeEmbedding(d_model)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, z: torch.Tensor, tau: torch.Tensor, modality_idx: int) -> torch.Tensor:
        mod_idx = torch.tensor(modality_idx, device=z.device, dtype=torch.long)
        e_m = self.modality_embeddings(mod_idx).view(1, 1, self.d_model)
        if not self.time_enabled:
            return self.norm(z + e_m)                       # HTT_TIME_EMBED_ENABLED = False
        return self.norm(z + e_m + self.time_embedding(tau))


class GridLayerNorm(nn.Module):
    """LayerNorm for the lag-bias grid (v18).

    g(Δτ) produces ONE scalar per (query i, key j) pair, so its output is a (B, T_m, T_n) grid with no feature
    dimension > 1. A LayerNorm over a size-1 dimension would subtract each value from itself and return 0 for every
    input. Instead, the normalization is taken over each window's whole (T_m, T_n) grid, i.e. every bias value that
    enters that window's attention matrix. It is computed separately for each window: every window has its own real
    timestamps, and mixing windows would make one window's bias depend on the rest of the batch and differ between
    training and evaluation. There are no affine parameters, so beta is the only source of scale.

    Consequences: the subtracted mean is one constant per window, and softmax is unchanged by a constant added to
    every score, so only the division by the std affects attention. For an all-zero grid (the zero-init state) the
    output is exactly 0. As g develops variance v, the output std grows smoothly as sqrt(v / (v + eps)) toward 1."""

    def __init__(self, eps: float = LAG_BIAS_LN_EPS):
        super().__init__()
        self.eps = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:   # x: (B, T_m, T_n)
        return F.layer_norm(x, x.shape[-2:], eps=self.eps)


class PairwiseLACA(nn.Module):
    """Cross-attention from modality m to n with a learned lag bias g(Δτ) added to the scores (scaled by β)."""

    def __init__(self, d_model: int, num_heads: int = LACA_NUM_HEADS, beta: float = LACA_BETA):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.scale = 1.0 / math.sqrt(self.head_dim)
        self.beta = beta
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.lag_bias = nn.Sequential(nn.Linear(1, LAG_BIAS_HIDDEN), nn.Tanh(), nn.Linear(LAG_BIAS_HIDDEN, 1))
        # v18 ZERO-INIT: g's final Linear starts at exactly 0 -> g(Δτ) = 0 for every Δτ and every seed. The layer was
        # already built (same RNG draws as v16), and nn.init.zeros_ draws no random numbers, so every other
        # layer's initial weights and the training RNG stream are unchanged.
        nn.init.zeros_(self.lag_bias[-1].weight)
        nn.init.zeros_(self.lag_bias[-1].bias)
        self.lag_bias_norm = GridLayerNorm()   # v18: no parameters

    def lag_bias_term(self, tau_m, tau_n):
        """LN_grid(g(Δτ)), BEFORE beta. Returns (B, 1, T_m, T_n), broadcast over the heads."""
        delta_tau = (tau_m.unsqueeze(2) - tau_n.unsqueeze(1)).unsqueeze(-1)   # (B, T_m, T_n, 1)
        g = self.lag_bias(delta_tau).squeeze(-1)                                # (B, T_m, T_n): one scalar per (i, j)
        return self.lag_bias_norm(g).unsqueeze(1)                               # normalized per window -> (B, 1, T_m, T_n)

    def forward(self, H_m, H_n, tau_m, tau_n):
        B, T_m, _ = H_m.shape
        _, T_n, _ = H_n.shape
        Q = self.w_q(H_m).view(B, T_m, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.w_k(H_n).view(B, T_n, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.w_v(H_n).view(B, T_n, self.num_heads, self.head_dim).transpose(1, 2)

        scores = torch.matmul(Q, K.transpose(-2, -1)) * self.scale
        lag_bias = self.lag_bias_term(tau_m, tau_n)

        attn = F.softmax(scores + self.beta * lag_bias, dim=-1)   # beta = 1.0 (lag bias ON) or exactly 0.0 (OFF)
        out = torch.matmul(attn, V).transpose(1, 2).contiguous().view(B, T_m, self.d_model)

        attn_mean = attn.mean(dim=1)
        tau_diff = tau_n.unsqueeze(1) - tau_m.unsqueeze(2)
        expected_lag = (attn_mean * tau_diff).sum(dim=(1, 2)) / float(T_m)
        return self.out_proj(out), expected_lag


class BiModalLACAFusion(nn.Module):
    def __init__(self, d_model: int, beta: float = LACA_BETA):
        super().__init__()
        self.laca_as = PairwiseLACA(d_model, beta=beta)   # beta scales the lag bias in BOTH directions
        self.laca_sa = PairwiseLACA(d_model, beta=beta)
        self.gamma_as = nn.Parameter(torch.tensor(0.5))
        self.gamma_sa = nn.Parameter(torch.tensor(0.5))

    def forward(self, Ha, Hs, tau_a, tau_s):
        H_a_from_s, lag_as = self.laca_as(Ha, Hs, tau_a, tau_s)
        H_s_from_a, lag_sa = self.laca_sa(Hs, Ha, tau_s, tau_a)
        return Ha + self.gamma_as * H_a_from_s, Hs + self.gamma_sa * H_s_from_a, {"lag_as": lag_as, "lag_sa": lag_sa}


class MambaBlock(nn.Module):
    """Pre-norm residual wrapper around the real mamba_ssm.Mamba selective-scan SSM."""

    def __init__(self, d_model: int, d_state: int, d_conv: int, expand: int):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.mamba = MAMBA_CLASS(d_model=d_model, d_state=d_state, d_conv=d_conv, expand=expand)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.mamba(self.norm(x))


class FallbackGatedConvBlock(nn.Module):
    """CPU-only stand-in when mamba_ssm is unavailable: depthwise causal conv + gated MLP. NOT a state-space
    model (no hidden state, no A/B/C, no discretization)."""

    def __init__(self, d_model: int, d_conv: int, expand: int):
        super().__init__()
        d_inner = expand * d_model
        self.in_proj = nn.Linear(d_model, d_inner * 2, bias=False)
        self.conv1d = nn.Conv1d(d_inner, d_inner, kernel_size=d_conv, padding=d_conv - 1, groups=d_inner, bias=True)
        self.mixer = nn.Sequential(nn.Linear(d_inner, d_inner), nn.GELU(), nn.Linear(d_inner, d_inner))
        self.out_proj = nn.Linear(d_inner, d_model, bias=False)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        L = x.shape[1]
        x_branch, z_branch = self.in_proj(self.norm(x)).chunk(2, dim=-1)
        x_conv = F.silu(self.conv1d(x_branch.transpose(1, 2))[:, :, :L].transpose(1, 2))
        return x + self.out_proj(self.mixer(x_conv) * F.silu(z_branch))


def build_fusion_blocks(d_model, num_blocks, d_state, d_conv, expand=MAMBA_EXPAND):
    if MAMBA_CLASS is not None:
        return nn.ModuleList([MambaBlock(d_model, d_state, d_conv, expand) for _ in range(num_blocks)])
    return nn.ModuleList([FallbackGatedConvBlock(d_model, d_conv, expand) for _ in range(num_blocks)])


# The reference results were produced by a model whose constructor drew, from the torch RNG, the initial weights
# of a 4-block (d_state 16, d_conv 4) Mamba stack before the final 2-block stack was built from the SAME starting
# RNG state. Every module built afterwards (pooling, classifier, auxiliary heads) and the whole training RNG stream
# (shuffle order, dropout masks) depend on the RNG state that 4-block draw leaves behind. To reproduce those
# results exactly, the constructor performs the same draw and discards it; the discarded blocks are never part of
# the model. This is a fixed step of weight initialization, not a configuration option.
RNG_REFERENCE_STACK = (4, 16, 4)   # (num_blocks, d_state, d_conv) whose initialization is drawn and discarded


class AeroSyncMamba(nn.Module):
    def __init__(self, d_model: int = D_MODEL, n_classes: int = N_CLASSES, htt_time: bool = True, laca_lag: bool = False):
        super().__init__()
        self.d_model = d_model
        self.n_classes = n_classes
        self.htt_time_enabled = htt_time                  # HTT_TIME_EMBED_ENABLED
        self.laca_lag_enabled = laca_lag                  # LACA_LAG_BIAS_ENABLED
        self.laca_beta = LACA_BETA if laca_lag else 0.0   # OFF = exactly 0.0 (v18's beta = 0 code path)

        self.sensor_encoder = nn.Sequential(
            nn.Conv1d(N_SENSOR_CHANNELS, 64, kernel_size=5, padding=2), nn.BatchNorm1d(64), nn.GELU(),
            nn.Conv1d(64, d_model, kernel_size=3, padding=1), nn.BatchNorm1d(d_model), nn.GELU(),
        )
        self.audio_encoder = nn.Sequential(
            nn.Linear(N_AUDIO_BINS, d_model), nn.LayerNorm(d_model), nn.GELU(), nn.Linear(d_model, d_model)
        )
        self.htt = HeterogeneousTemporalTokenization(d_model, time_enabled=htt_time)
        self.laca = BiModalLACAFusion(d_model, beta=self.laca_beta)   # plain float: no params, no RNG draws
        self.sep_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)

        rng_before = torch.get_rng_state()
        build_fusion_blocks(d_model, *RNG_REFERENCE_STACK)
        rng_after_reference = torch.get_rng_state()
        torch.set_rng_state(rng_before)
        self.mamba_blocks = build_fusion_blocks(d_model, MAMBA_NUM_BLOCKS, MAMBA_D_STATE, MAMBA_D_CONV)
        torch.set_rng_state(rng_after_reference)

        self.pool_attn = nn.Sequential(nn.Linear(d_model, 64), nn.Tanh(), nn.Linear(64, 1))
        self.classifier = nn.Sequential(
            nn.Dropout(DROPOUT),
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Linear(d_model // 2, n_classes),
        )
        self.aux_a = nn.Linear(d_model, n_classes)
        self.aux_s = nn.Linear(d_model, n_classes)

    def forward(self, xa, xs, ta, ts):
        B = xa.shape[0]
        za = self.audio_encoder(xa)
        zs = self.sensor_encoder(xs.transpose(1, 2)).transpose(1, 2)

        Ha = self.htt(za, ta, modality_idx=0)
        Hs = self.htt(zs, ts, modality_idx=1)
        H_tilde_a, H_tilde_s, lags = self.laca(Ha, Hs, ta, ts)
        za_p = H_tilde_a.mean(dim=1)
        zs_p = H_tilde_s.mean(dim=1)

        F_seq = torch.cat([H_tilde_a, self.sep_token.expand(B, -1, -1), H_tilde_s], dim=1)
        for block in self.mamba_blocks:
            F_seq = block(F_seq)

        weights = F.softmax(self.pool_attn(F_seq), dim=1)
        zf = (F_seq * weights).sum(dim=1)
        logits = self.classifier(zf)

        pooled = {"z_a": za_p, "z_s": zs_p}
        aux_logits = {"y_a": self.aux_a(za_p), "y_s": self.aux_s(zs_p)}
        return logits, pooled, aux_logits, lags


def verify_architecture(model):
    """Finalized architecture, read from the constructed modules, including both v22 switches."""
    assert model.htt.time_enabled == model.htt_time_enabled
    beta = LACA_BETA if model.laca_lag_enabled else 0.0
    for name in ("laca_as", "laca_sa"):
        laca = getattr(model.laca, name)
        assert laca.num_heads == LACA_NUM_HEADS, f"{name}: {laca.num_heads} heads"
        assert laca.beta == beta, f"{name}: beta {laca.beta} != {beta} for LACA_LAG_BIAS_ENABLED={model.laca_lag_enabled}"
        assert sum(p.numel() for p in laca.lag_bias.parameters()) == LAG_BIAS_PARAMS
        assert isinstance(laca.lag_bias_norm, GridLayerNorm) and not list(laca.lag_bias_norm.parameters())
    assert len(model.mamba_blocks) == MAMBA_NUM_BLOCKS
    mamba_params = sum(p.numel() for p in model.mamba_blocks.parameters())
    if MAMBA_CLASS is not None:
        for block in model.mamba_blocks:
            m = block.mamba
            assert (m.d_state, m.d_conv, m.expand) == (MAMBA_D_STATE, MAMBA_D_CONV, MAMBA_EXPAND)
        assert mamba_params == EXPECTED_MAMBA_PARAMS, f"Mamba params {mamba_params:,} != {EXPECTED_MAMBA_PARAMS:,}"
    return mamba_params


_probe = AeroSyncMamba(htt_time=HTT_TIME_EMBED_ENABLED, laca_lag=LACA_LAG_BIAS_ENABLED)
print(f"Model (Condition {CONDITION}): {sum(p.numel() for p in _probe.parameters()):,} params | fusion blocks: "
      f"{type(_probe.mamba_blocks[0]).__name__} x {len(_probe.mamba_blocks)} ({verify_architecture(_probe):,} params) | "
      f"HTT time embedding {'ON' if _probe.htt.time_enabled else 'OFF'} | LACA beta {_probe.laca.laca_as.beta} / "
      f"{_probe.laca.laca_sa.beta} | backbone: {FUSION_BACKBONE}")
assert not _probe.htt.time_enabled and _probe.laca.laca_as.beta == 0.0 and _probe.laca.laca_sa.beta == 0.0
del _probe

## 5. Loss, metrics and the training function
This is v24's cell, unchanged apart from one thing: `train_one_run(seed, bundle, run_name)` takes the dataset bundle
and a run name. The same function therefore trains both the gate (v24's split) and every fold. The rest is v24's:
- class-weighted CE + 0.1 InfoNCE + 0.1 KL;
- AdamW 1e-4 (weight decay 1e-4) with cosine decay to 1e-6, gradient clip 1.0;
- 20 epochs, batch 32, and the best epoch by validation macro-F1.

The per-run checks are also v24's: initial weights and RNG against the seed's reference, both switches exactly 0, and
τ-invariance.

**v28:** unchanged from v27; in v28 it is used only for the Stage-0 fused reproduction gate.

In [ ]:
# ==============================================================================
# v27: v24's training cell, unchanged except that train_one_run takes the dataset BUNDLE and a run name (so the same
# function trains the reproduction gate on v24's split and every cross-day fold). Loss, optimizer, schedule, epochs,
# batch size, checkpoint selection (best validation macro-F1) and the RNG-consuming order are v24's.
# TRAINING — loss (unchanged), metrics (4 standard + confusion matrix + Class 3/4 recall), switch checks and
# train_one_run(seed, bundle, run_name).
# ==============================================================================
from sklearn.metrics import confusion_matrix


class AeroSyncLoss(nn.Module):
    """L = CE_classweighted(logits, y) + LAMBDA_ALIGN * InfoNCE(z_a, z_s) + LAMBDA_CONS * KL-consistency."""

    def __init__(self, class_weights):
        super().__init__()
        self.lambda_align = LAMBDA_ALIGN
        self.lambda_cons = LAMBDA_CONS
        self.temperature = INFONCE_TEMPERATURE
        self.ce = nn.CrossEntropyLoss(weight=class_weights)

    def _infonce(self, z1, z2):
        sim = torch.matmul(F.normalize(z1, p=2, dim=-1), F.normalize(z2, p=2, dim=-1).t()) / self.temperature
        labels = torch.arange(z1.shape[0], device=z1.device)
        return 0.5 * (F.cross_entropy(sim, labels) + F.cross_entropy(sim.t(), labels))

    def forward(self, logits, targets, pooled, aux_logits):
        loss_cls = self.ce(logits, targets)
        loss_align = self._infonce(pooled["z_a"], pooled["z_s"])
        p_fused = F.softmax(logits.detach(), dim=-1)
        loss_cons = (
            F.kl_div(F.log_softmax(aux_logits["y_a"], dim=-1), p_fused, reduction="batchmean") +
            F.kl_div(F.log_softmax(aux_logits["y_s"], dim=-1), p_fused, reduction="batchmean")
        ) / 2.0
        total = loss_cls + self.lambda_align * loss_align + self.lambda_cons * loss_cons
        return total, {"loss_cls": loss_cls.item(), "loss_align": loss_align.item(), "loss_cons": loss_cons.item()}


def compute_metrics(y_true, y_prob):
    """accuracy, macro-F1, balanced accuracy, macro one-vs-rest AUROC from class probabilities (prediction = argmax)."""
    y_pred = y_prob.argmax(axis=1)
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_auroc": float(roc_auc_score(y_true, y_prob, multi_class="ovr", average="macro")),
    }


def evaluate_probs(y_true, y_prob):
    """4 standard metrics + full 5x5 confusion matrix + per-class recall + Class 3/4 recall + 3->4 / 4->3 counts."""
    cm = confusion_matrix(y_true, y_prob.argmax(axis=1), labels=list(range(N_CLASSES)))
    recall = cm.diagonal() / np.maximum(cm.sum(axis=1), 1)
    return {**compute_metrics(y_true, y_prob), "confusion_matrix": cm,
            **{f"recall_c{c}": float(recall[c]) for c in range(N_CLASSES)},
            "class3_recall": float(recall[3]), "class4_recall": float(recall[4]),
            "err_3to4": int(cm[3, 4]), "err_4to3": int(cm[4, 3])}


def print_confusion(ev, tag):
    cm = ev["confusion_matrix"]
    print(f"[CONFUSION {tag}] rows = TRUE severity, cols = PREDICTED severity (test set, {int(cm.sum())} windows)")
    print("            " + "".join(f"pred {j:>2} " for j in range(N_CLASSES)) + "|  recall")
    for i in range(N_CLASSES):
        print(f"   true {i:>2} " + "".join(f"{int(cm[i, j]):>8}" for j in range(N_CLASSES)) + f"  |  {ev[f'recall_c{i}']:.4f}")
    print(f"[CLASS 3/4 {tag}] Class 3 recall {ev['class3_recall']:.4f} | Class 4 recall {ev['class4_recall']:.4f} | "
          f"3->4 (true 3, pred 4) {ev['err_3to4']} | 4->3 (true 4, pred 3) {ev['err_4to3']}")


def train_class_weights(windows):
    """Inverse class frequency over the TRAINING windows."""
    counts = np.bincount([w["label"] for w in windows], minlength=N_CLASSES)
    return torch.tensor(counts.sum() / (len(counts) * np.maximum(counts, 1)), dtype=torch.float32)


def _batch_to_device(batch, non_blocking=False):
    return tuple(batch[k].to(device, non_blocking=non_blocking) for k in ("x_a", "x_s", "tau_a", "tau_s", "label"))


def predict_probs(model, loader, tau_shift=0.0):
    """Softmax probabilities and true labels over a loader (eval mode, no grad), in loader order.
    tau_shift != 0 adds the shift to BOTH tau_a and tau_s (used only for the Condition D invariance check)."""
    model.eval()
    probs, labels = [], []
    with torch.no_grad():
        for batch in loader:
            xa, xs, ta, ts, y = _batch_to_device(batch)
            logits, _, _, _ = model(xa, xs, ta + tau_shift, ts + tau_shift) if tau_shift else model(xa, xs, ta, ts)
            probs.append(F.softmax(logits, dim=-1).cpu().numpy())
            labels.append(y.cpu().numpy())
    return np.concatenate(probs), np.concatenate(labels)


METRICS = ["accuracy", "macro_f1", "balanced_accuracy", "macro_auroc"]


def make_bundle(data):
    """Loaders, class weights and probe batches for one dataset — built exactly as v22 built its globals."""
    tr = data["train_ds"]
    return {**data,
            "train_loader": DataLoader(tr, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True, num_workers=NUM_WORKERS),
            "val_loader": DataLoader(data["val_ds"], batch_size=BATCH_SIZE, shuffle=False, pin_memory=True, num_workers=NUM_WORKERS),
            "test_loader": DataLoader(data["test_ds"], batch_size=BATCH_SIZE, shuffle=False, pin_memory=True, num_workers=NUM_WORKERS),
            "class_weights": train_class_weights(data["split_windows"]["train"]).to(device),
            "test_labels": np.array([w["label"] for w in data["split_windows"]["test"]], dtype=np.int64),
            "test_missions": np.array(data["window_missions"]["test"]),
            # fixed probe batches, read by index (no DataLoader, no RNG)
            "lag_probe_taus": tuple(torch.stack([tr[i][k] for i in range(min(BATCH_SIZE, len(tr)))]) for k in ("tau_a", "tau_s")),
            "switch_probe_idx": np.linspace(0, len(tr) - 1, BATCH_SIZE).astype(int)}


# v27: bundles are built per use (the gate on v24's split, then one per fold) — see the gate and fold-run cells.


def lag_bias_contribution(model, bundle):
    """Per LACA direction: max|beta * LN_grid(g(Δτ))|, the exact term PairwiseLACA adds to the attention scores."""
    ta, ts = (t.to(device) for t in bundle["lag_probe_taus"])
    out = {}
    with torch.no_grad():
        for name, laca, tau_m, tau_n in (("laca_as", model.laca.laca_as, ta, ts), ("laca_sa", model.laca.laca_sa, ts, ta)):
            contrib = laca.beta * laca.lag_bias_term(tau_m, tau_n)
            out[name] = {"beta": laca.beta, "contrib_absmax": float(contrib.abs().max()),
                         "contrib_finite": bool(torch.isfinite(contrib).all())}
    return out


def htt_time_contribution(model, bundle):
    """What HTT's time embedding contributes on a fixed real batch (eval mode, RNG untouched):
    term = max|HTT(z, tau) - norm(z + e_m)| and tau_sens = max|HTT(z, tau) - HTT(z, tau + 0.3)|, per modality."""
    items = [bundle["train_ds"][int(i)] for i in bundle["switch_probe_idx"]]
    xa, xs, ta, ts = (torch.stack([it[k] for it in items]).to(device) for k in ("x_a", "x_s", "tau_a", "tau_s"))
    was_training = model.training
    cuda_devices = list(range(torch.cuda.device_count())) if torch.cuda.is_available() else []
    out = {}
    with torch.random.fork_rng(devices=cuda_devices), torch.no_grad():
        model.eval()
        za = model.audio_encoder(xa)
        zs = model.sensor_encoder(xs.transpose(1, 2)).transpose(1, 2)
        for name, z, tau, idx in (("audio", za, ta, 0), ("sensor", zs, ts, 1)):
            e_m = model.htt.modality_embeddings(torch.tensor(idx, device=device)).view(1, 1, model.d_model)
            actual = model.htt(z, tau, modality_idx=idx)
            out[name] = {"term": float((actual - model.htt.norm(z + e_m)).abs().max()),
                         "tau_sens": float((actual - model.htt(z, tau + 0.3, modality_idx=idx)).abs().max())}
    model.train(was_training)
    return out


def check_switches(model, bundle, tag, when):
    """Condition D: both temporal contributions must be EXACTLY 0."""
    h = htt_time_contribution(model, bundle)
    lag = lag_bias_contribution(model, bundle)
    h_max = max(max(v["term"], v["tau_sens"]) for v in h.values())
    lag_max = max(r["contrib_absmax"] for r in lag.values())
    assert h_max == 0.0, f"HTT time embedding is OFF but contributes {h_max!r}"
    assert all(r["contrib_finite"] and r["contrib_absmax"] == 0.0 for r in lag.values()), f"LACA lag OFF but {lag}"
    print(f"{tag} [SWITCH CHECK {when}] ✅ HTT time term {h_max!r}, LACA lag term {lag_max!r} — both EXACTLY 0")


def train_one_run(seed: int, bundle: dict, run_name: str):
    """Train Condition D with `seed` on `bundle` (train/val/test of one split), evaluate on its test set.
    Returns (trained model, result dict incl. test probabilities and confusion matrix)."""
    t0 = time.time()
    assert len(bundle["test_labels"]) == len(bundle["test_missions"]) == len(bundle["test_ds"])
    tag = f"[{run_name}]"
    set_all_seeds(seed)
    model = AeroSyncMamba(htt_time=HTT_TIME_EMBED_ENABLED, laca_lag=LACA_LAG_BIAS_ENABLED).to(device)
    init_sha1 = hashlib.sha1(b"".join(v.detach().cpu().contiguous().numpy().tobytes()
                                      for v in model.state_dict().values())).hexdigest()
    rng_sha1 = hashlib.sha1(torch.get_rng_state().numpy().tobytes()).hexdigest()
    mamba_params = verify_architecture(model)
    ref_init, ref_rng = SEED_INIT_REFERENCE[seed]
    paired_ok = init_sha1[:12] == ref_init and rng_sha1[:12] == ref_rng
    print(f"{tag} HTT_TIME_EMBED_ENABLED = {model.htt.time_enabled} | LACA_LAG_BIAS_ENABLED = {model.laca_lag_enabled} "
          f"(beta {model.laca.laca_as.beta}) | fusion {type(model.mamba_blocks[0]).__name__} ({mamba_params:,} params) | "
          f"train {len(bundle['train_ds'])} / val {len(bundle['val_ds'])} / test {len(bundle['test_ds'])} windows")
    print(f"{tag} [PAIRING] initial weights {init_sha1[:12]} vs reference {ref_init} | RNG {rng_sha1[:12]} vs reference "
          f"{ref_rng} -> " + ("✅ identical start (same for every run of this seed)" if paired_ok else
                              "🚨 MISMATCH — this seed's runs are not cleanly paired"))
    check_switches(model, bundle, tag, "before training")

    criterion = AeroSyncLoss(bundle["class_weights"])
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR_MIN)

    best_val_f1, best_state, best_epoch = 0.0, None, None
    for epoch in range(1, EPOCHS + 1):
        model.train()
        running = defaultdict(float)
        for batch in tqdm(bundle["train_loader"], desc=f"{run_name} epoch {epoch:02d}/{EPOCHS}", leave=False):
            xa, xs, ta, ts, y = _batch_to_device(batch, non_blocking=True)
            optimizer.zero_grad()
            logits, pooled, aux_logits, _ = model(xa, xs, ta, ts)
            loss, parts = criterion(logits, y, pooled, aux_logits)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP_NORM)
            optimizer.step()
            running["loss"] += loss.item()
            for k, v in parts.items():
                running[k] += v
        scheduler.step()

        lag = lag_bias_contribution(model, bundle)
        assert all(r["contrib_finite"] and r["contrib_absmax"] == 0.0 for r in lag.values()), f"lag leak at epoch {epoch}"

        val_probs, val_labels = predict_probs(model, bundle["val_loader"])
        val_f1 = compute_metrics(val_labels, val_probs)["macro_f1"]
        improved = val_f1 > best_val_f1
        if improved:
            best_val_f1, best_state, best_epoch = val_f1, copy.deepcopy(model.state_dict()), epoch
        n = len(bundle["train_loader"])
        print(f"{tag} epoch {epoch:02d}/{EPOCHS} | lag term 0 ✅ | train loss {running['loss'] / n:.4f} (cls "
              f"{running['loss_cls'] / n:.4f}, align {running['loss_align'] / n:.4f}, cons {running['loss_cons'] / n:.4f}) "
              f"| val macro-F1 {val_f1:.4f}{'  * best' if improved else ''}")

    model.load_state_dict(best_state)
    check_switches(model, bundle, tag, f"best checkpoint, epoch {best_epoch}")
    test_probs, test_labels = predict_probs(model, bundle["test_loader"])
    assert np.array_equal(test_labels, bundle["test_labels"])
    ev = evaluate_probs(test_labels, test_probs)
    shifted_probs, _ = predict_probs(model, bundle["test_loader"], tau_shift=0.3)
    tau_invariant = bool(np.array_equal(shifted_probs, test_probs))
    print(f"{tag} [TAU INVARIANCE] test predictions with every tau_a / tau_s shifted by +0.3: "
          + ("✅ bit-identical — Condition D uses no timestamps" if tau_invariant else "🚨 CHANGED — a timestamp path is active in Condition D"))
    torch.save(best_state, CHECKPOINT_DIR / f"{run_name}_best.pt")
    np.save(PROBS_DIR / f"{run_name}_test_probs.npy", test_probs)
    np.save(PROBS_DIR / f"{run_name}_test_labels.npy", test_labels)

    result = {
        "run_name": run_name, "condition": CONDITION, "seed": seed,
        **ev,
        "n_missions": bundle["n_missions"],
        "n_train_windows": len(bundle["train_ds"]),
        "n_test_windows": len(test_labels),
        "n_test_missions": len(bundle["split_missions"]["test"]),
        "best_val_macro_f1": round(best_val_f1, 6),
        "best_epoch": best_epoch,
        "init_sha1": init_sha1[:12],
        "rng_sha1": rng_sha1[:12],
        "paired_init": paired_ok,
        "tau_invariant": tau_invariant,
        "train_seconds": round(time.time() - t0, 1),
        "test_probs": test_probs,
        "test_labels": test_labels,
    }
    return model, result

## 6. Stage 0 — reproduction gate (fused Condition D, v27's code unchanged)
- **(a) Data.** On v24's original split:
  - the CORRECTED code sha1 must equal v23's;
  - the mission, window and class counts must equal v24's;
  - the strongest-64 bins must equal v24's;
  - the window-content sha1 must equal the value v27 printed.
- **(b) Training.** Fused D seed 42 on v24's split must give macro-F1 0.9014 and accuracy 0.9080, both ±0.0005.

On FAIL the cell prints a warning and **raises**, so no unimodal run starts. This cell also sets up the resumable CSV. On
a resumed session, the CSV and the `v28_test_probs/*.npz` files from an attached earlier output are copied back in.

In [ ]:
# ==============================================================================
# STAGE 0 — REPRODUCTION GATE with v27's FUSED pipeline (unchanged): (a) the original split's data == v24,
# (b) fused Condition D seed 42 on v24's split reproduces v24. STOPS on FAIL. Also: the resumable results log.
# ==============================================================================
from sklearn.metrics import precision_recall_fscore_support

RESULTS_FIELDS = (["stage", "arm", "protocol", "fold", "fold_day", "seed"] + METRICS
                  + [f"{m}_c{c}" for m in ("precision", "recall", "f1") for c in range(N_CLASSES)]
                  + ["confusion_matrix", "n_test_windows", "n_test_missions", "n_train_windows", "n_val_windows",
                     "window_ids_sha1", "n_params", "best_epoch", "best_val_macro_f1", "train_seconds", "probs_file",
                     "gate_check", "finished_at"])

if not RESULTS_CSV.exists():
    _prev = sorted(h for g in RESUME_CSV_GLOBS for h in glob.glob(g, recursive=True))
    if _prev:
        shutil.copyfile(_prev[0], RESULTS_CSV)
        print(f"[RESUME] copied an earlier session's results from {_prev[0]} -> {RESULTS_CSV}")
_n_copied = 0
for _g in RESUME_NPZ_GLOBS:
    for _h in glob.glob(_g, recursive=True):
        if not (PROBS_DIR / Path(_h).name).exists():
            shutil.copyfile(_h, PROBS_DIR / Path(_h).name)
            _n_copied += 1
if _n_copied:
    print(f"[RESUME] copied {_n_copied} per-run probability files into {PROBS_DIR}")


def load_rows():
    if not RESULTS_CSV.exists():
        return []
    with open(RESULTS_CSV, newline="") as f:
        return list(csv.DictReader(f))


def done_keys():
    """(arm, protocol, fold, seed) -> the last logged row for that run."""
    return {(r["arm"], r["protocol"], int(r["fold"]), int(r["seed"])): r for r in load_rows()}


def per_class_prf(y_true, probs):
    p, r, f, _ = precision_recall_fscore_support(y_true, probs.argmax(axis=1), labels=list(range(N_CLASSES)), zero_division=0)
    return {**{f"precision_c{c}": float(p[c]) for c in range(N_CLASSES)}, **{f"recall_c{c}": float(r[c]) for c in range(N_CLASSES)},
            **{f"f1_c{c}": float(f[c]) for c in range(N_CLASSES)}}


def log_row(row):
    """Append ONE finished run to the results CSV immediately (header written once)."""
    write_header = not RESULTS_CSV.exists()
    row = {**{k: row.get(k) for k in RESULTS_FIELDS}, "finished_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
    with open(RESULTS_CSV, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=RESULTS_FIELDS)
        if write_header:
            w.writeheader()
        w.writerow(row)


def mission_day(name):
    """Recording day from 'rosbag2_YYYY_MM_DD-HH_MM_SS' as 'YYYY-MM-DD', with the 2021-12-29 -> 2021-12-31 merge."""
    m = re.match(r"rosbag2_(\d{4})_(\d{2})_(\d{2})-", name)
    assert m, f"cannot read a date from mission name {name!r}"
    d = f"{m.group(1)}-{m.group(2)}-{m.group(3)}"
    return DAY_MERGE.get(d, d)


def free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def _windows_sha1(windows):
    h = hashlib.sha1()
    for w in windows:
        for k in ("x_a", "x_s", "tau_a", "tau_s"):
            h.update(np.ascontiguousarray(w[k]).tobytes())
        h.update(str(w["label"]).encode())
    return h.hexdigest()[:16]


# --- (a) the original split's data == v24 (and == v27's recorded window-content sha1) ---
V27_WINDOWS_SHA1 = {"train": "802bfe02e34f38d2", "val": "0d5f6a86377ff63d", "test": "48793996ffcfa913"}   # v27 run output
GATE_WINDOWS_SHA1 = {s: _windows_sha1(CORRECTED_ALIGNMENT["split_windows"][s]) for s in ("train", "val", "test")}
GATE_DATA_CHECK = {
    "CORRECTED code sha1 == v23/v24 (parse + slice)": False not in CORRECTED_SOURCE_CHECK.values(),
    "missions / windows / class counts == v24": all(CORRECTED_DATA_CHECK.values()),
    "strongest-64 bins == v24's (= v18 reference indices)": CORRECTED_ALIGNMENT["bins"].tolist() == REFERENCE_DATA["bin_indices"],
    "window-content sha1 == v27's (train/val/test)": GATE_WINDOWS_SHA1 == V27_WINDOWS_SHA1,
}
print("█" * 110)
print("█  STAGE 0 — REPRODUCTION GATE (a): original v24 split, data")
for _name, _ok in GATE_DATA_CHECK.items():
    print(f"█    {_name:<56}: {'✅ PASS' if _ok else '🚨 FAIL'}")
print(f"█    window-content sha1 now {GATE_WINDOWS_SHA1}")
GATE_A_PASS = all(GATE_DATA_CHECK.values())

# --- (b) FUSED Condition D seed 42 on v24's split (v27's train_one_run, unchanged) ---
_prev_gate = done_keys().get(("FUSED_D", "standard", 0, 42))
if _prev_gate is not None and _prev_gate["gate_check"] == "PASS":
    GATE_RESULT = {k: float(_prev_gate[k]) for k in METRICS}
    GATE_RESULT["confusion_matrix"] = np.array(json.loads(_prev_gate["confusion_matrix"]))
    GATE_SOURCE = "read from the results CSV (already passed in an earlier session)"
else:
    BUNDLE_ORIGINAL = make_bundle(CORRECTED_ALIGNMENT)
    _model, GATE_RESULT = train_one_run(42, BUNDLE_ORIGINAL, "gate_original_FUSED_D_seed42")
    GATE_SOURCE = "trained now"
    del _model, BUNDLE_ORIGINAL
    free_gpu()
GATE_DELTA = {k: GATE_RESULT[k] - V24_CORRECTED_D_SEED42[k] for k in ("macro_f1", "accuracy")}
GATE_CM_SAME = bool(np.array_equal(np.asarray(GATE_RESULT["confusion_matrix"]), np.array(V24_CORRECTED_D_SEED42["confusion_matrix"])))
GATE_B_PASS = all(abs(v) <= GATE_TOLERANCE for v in GATE_DELTA.values())
GATE_PASS = GATE_A_PASS and GATE_B_PASS
if _prev_gate is None or _prev_gate["gate_check"] != "PASS":
    log_row({"stage": "0", "arm": "FUSED_D", "protocol": "standard", "fold": 0, "fold_day": "original v24 split", "seed": 42,
             **{k: GATE_RESULT[k] for k in METRICS}, **per_class_prf(GATE_RESULT["test_labels"], GATE_RESULT["test_probs"]),
             "confusion_matrix": json.dumps(np.asarray(GATE_RESULT["confusion_matrix"]).tolist()),
             "n_test_windows": GATE_RESULT["n_test_windows"], "best_epoch": GATE_RESULT["best_epoch"],
             "best_val_macro_f1": GATE_RESULT["best_val_macro_f1"], "train_seconds": GATE_RESULT["train_seconds"],
             "gate_check": "PASS" if GATE_PASS else "FAIL"})
print("█  STAGE 0 — REPRODUCTION GATE (b): FUSED Condition D seed 42 on v24's split (" + GATE_SOURCE + ")")
print(f"█    macro-F1 {GATE_RESULT['macro_f1']:.4f} vs v24 {V24_CORRECTED_D_SEED42['macro_f1']:.4f} (Δ {GATE_DELTA['macro_f1']:+.5f}) | "
      f"accuracy {GATE_RESULT['accuracy']:.4f} vs v24 {V24_CORRECTED_D_SEED42['accuracy']:.4f} (Δ {GATE_DELTA['accuracy']:+.5f}) | "
      f"confusion matrix {'identical' if GATE_CM_SAME else 'DIFFERENT'} | tolerance ±{GATE_TOLERANCE} -> "
      + ("✅ PASS" if GATE_B_PASS else "🚨 FAIL"))
print("█" * 110)
if GATE_PASS:
    print("✅ STAGE 0 GATE PASSED — the data and training code are the ones that produced v24/v27's numbers.")
else:
    print("\n" + "🚨" * 50)
    print("🚨 STAGE 0 GATE FAILED — " + ("the original split's data differs from v24/v27" if not GATE_A_PASS else
                                        "fused D seed 42 does not reproduce v24's macro-F1 / accuracy"))
    print("🚨 The unimodal arms would not share v24/v27's data pipeline. STOPPING before any unimodal run.")
    print("🚨" * 50)
    raise RuntimeError("v28 Stage-0 reproduction gate FAILED — see the warning above")

## 7. Folds — days, day-out, matched-random, inner validation
This cell makes no model calls. It:
- prints the day × class table and checks that every class is on every day;
- builds both partitions and asserts that their per-class counts are identical;
- splits off each fold's inner validation set;
- asserts there is no train/val/test overlap, and no test day in day-out training;
- prints both fold tables: missions per class, mission and window counts, and how many matched-random test missions share
  a day with their training missions.

In [ ]:
# ==============================================================================
# FOLDS — recording days, DAY-OUT and MATCHED-RANDOM partitions, inner validation, leakage asserts, fold tables
# ==============================================================================
ALL_MISSIONS = sorted((m for s in corrected_splits for lab in corrected_splits[s] for m in corrected_splits[s][lab]),
                      key=lambda m: m.name)
LABEL = {m: mission_home[m][1] for m in ALL_MISSIONS}
DAY = {m: mission_day(m.name) for m in ALL_MISSIONS}
assert len(ALL_MISSIONS) == len(set(ALL_MISSIONS)) == V23_CORRECTED_DATA["n_missions"], f"{len(ALL_MISSIONS)} missions"
_merged = [m.name for m in ALL_MISSIONS if m.name[8:18].replace("_", "-") in DAY_MERGE]
print(f"[DAYS] merged into another day's group ({DAY_MERGE}): {_merged} (class "
      f"{[LABEL[m] for m in ALL_MISSIONS if m.name in _merged]}) — expected exactly one mission")
assert len(_merged) == 1, f"expected ONE remaining 2021-12-29 mission, found {_merged}"
assert set(DAY.values()) == set(DAYS), f"days found {sorted(set(DAY.values()))} != {DAYS}"
# windows per mission do not depend on the audio bins (the grid and its skip rule use timestamps only)
WINDOWS_PER_MISSION = defaultdict(int)
for _s in ("train", "val", "test"):
    for _n in CORRECTED_ALIGNMENT["window_missions"][_s]:
        WINDOWS_PER_MISSION[_n] += 1

print(f"\n[DAYS x CLASSES] missions per recording day and class (all {len(ALL_MISSIONS)} CORRECTED missions)")
print(f"    {'day':<12}" + "".join(f"{'C' + str(c):>6}" for c in range(N_CLASSES)) + f"{'total':>8}{'windows':>9}")
DAY_CLASS = {d: np.bincount([LABEL[m] for m in ALL_MISSIONS if DAY[m] == d], minlength=N_CLASSES) for d in DAYS}
for d in DAYS:
    print(f"    {d:<12}" + "".join(f"{n:>6}" for n in DAY_CLASS[d]) + f"{DAY_CLASS[d].sum():>8}"
          f"{sum(WINDOWS_PER_MISSION[m.name] for m in ALL_MISSIONS if DAY[m] == d):>9}"
          + ("   <- SMALL fold" if d == SMALL_FOLD_DAY else ""))
_missing = {d: [c for c in range(N_CLASSES) if DAY_CLASS[d][c] == 0] for d in DAYS}
assert not any(_missing.values()), f"a class is missing from a recording day: {_missing} — per-fold macro-F1/AUROC undefined"
print("    every class is recorded on every one of the 5 days ✅")


def _split_inner_val(pool, protocol_idx, fold_idx):
    """~INNER_VAL_FRACTION of each class's training missions (min 1) -> validation, mission-level, fixed seed."""
    rng = np.random.default_rng([INNER_VAL_SEED, protocol_idx, fold_idx])
    train, val = [], []
    for c in range(N_CLASSES):
        ms = sorted((m for m in pool if LABEL[m] == c), key=lambda m: m.name)
        perm = rng.permutation(len(ms))
        n_val = max(1, int(round(INNER_VAL_FRACTION * len(ms))))
        assert len(ms) - n_val >= 1, f"class {c}: {len(ms)} training missions, cannot hold out {n_val}"
        val += [ms[i] for i in perm[:n_val]]
        train += [ms[i] for i in perm[n_val:]]
    return train, val


# --- DAY-OUT: fold k tests on every mission of day k ---
TEST_SETS = {("day_out", k): [m for m in ALL_MISSIONS if DAY[m] == d] for k, d in enumerate(DAYS)}
# --- MATCHED-RANDOM: per class, one fixed permutation cut into 5 folds with the day folds' per-class counts ---
_rng = np.random.default_rng(PARTITION_SEED)
_mr = {k: [] for k in range(len(DAYS))}
for c in range(N_CLASSES):
    ms = sorted((m for m in ALL_MISSIONS if LABEL[m] == c), key=lambda m: m.name)
    perm, pos = _rng.permutation(len(ms)), 0
    for k, d in enumerate(DAYS):
        n = int(DAY_CLASS[d][c])
        _mr[k] += [ms[i] for i in perm[pos:pos + n]]
        pos += n
    assert pos == len(ms)
for k in range(len(DAYS)):
    TEST_SETS[("matched_random", k)] = sorted(_mr[k], key=lambda m: m.name)
for k in range(len(DAYS)):
    a = np.bincount([LABEL[m] for m in TEST_SETS[("day_out", k)]], minlength=N_CLASSES)
    b = np.bincount([LABEL[m] for m in TEST_SETS[("matched_random", k)]], minlength=N_CLASSES)
    assert np.array_equal(a, b), f"fold {k}: per-class counts differ (day-out {a.tolist()} vs matched-random {b.tolist()})"
print(f"\n[MATCHED-RANDOM] partition seed {PARTITION_SEED}: every fold has EXACTLY the day fold's per-class mission counts ✅")

FOLDS = {}
for pi, p in enumerate(PROTOCOLS):
    assert sorted((m for k in range(len(DAYS)) for m in TEST_SETS[(p, k)]), key=lambda m: m.name) == ALL_MISSIONS, \
        f"{p}: the 5 test folds do not partition the 95 missions"
    for k in range(len(DAYS)):
        test = TEST_SETS[(p, k)]
        train, val = _split_inner_val([m for m in ALL_MISSIONS if m not in set(test)], pi, k)
        tr, va, te = set(train), set(val), set(test)
        assert not (tr & va) and not (tr & te) and not (va & te), f"{p} fold {k}: a mission is in two of train/val/test"
        assert tr | va | te == set(ALL_MISSIONS)
        if p == "day_out":
            assert not ({DAY[m] for m in train + val} & {DAY[m] for m in test}), f"day-out fold {k}: the test day is in train/val"
        FOLDS[(p, k)] = {"train": train, "val": val, "test": test,
                         "fold_day": DAYS[k] if p == "day_out" else f"matched to {DAYS[k]}"}
print("[LEAKAGE] every fold: no mission in more than one of train/val/test ✅ | day-out: no test day in train or val ✅")


def _n_windows(ms):
    return sum(WINDOWS_PER_MISSION[m.name] for m in ms)


for p in PROTOCOLS:
    print("\n" + "=" * 128 + f"\nFOLD TABLE — {p.upper()}\n" + "=" * 128)
    print(f"    {'fold':<5}{'test':<24}{'test missions per class':<26}{'missions tr/val/test':<22}{'windows tr/val/test':<22}"
          f"{'test missions sharing a day with train':<40}")
    for k in range(len(DAYS)):
        f = FOLDS[(p, k)]
        tr_days = {DAY[m] for m in f["train"]}
        shared = sum(DAY[m] in tr_days for m in f["test"])
        per_class = str(np.bincount([LABEL[m] for m in f["test"]], minlength=N_CLASSES).tolist())
        n_m = "/".join(str(len(f[s])) for s in ("train", "val", "test"))
        n_w = "/".join(str(_n_windows(f[s])) for s in ("train", "val", "test"))
        small = "   <- SMALL fold (10 missions)" if DAYS[k] == SMALL_FOLD_DAY else ""
        print(f"    {k:<5}{f['fold_day']:<24}{per_class:<26}{n_m:<22}{n_w:<22}{shared}/{len(f['test'])}{small}")
        if p == "matched_random":
            print(f"          test missions by day: " + ", ".join(
                f"{d} {sum(DAY[m] == d for m in f['test'])}" for d in DAYS if any(DAY[m] == d for m in f["test"])))
MR_SHARED_TOTAL = sum(sum(DAY[m] in {DAY[x] for x in FOLDS[("matched_random", k)]["train"]} for m in FOLDS[("matched_random", k)]["test"])
                      for k in range(len(DAYS)))
print(f"\n[MATCHED-RANDOM] {MR_SHARED_TOTAL} of {len(ALL_MISSIONS)} test missions share a recording day with that fold's "
      f"training missions (expected: most) | day-out: 0 by construction")

## 8. Unimodal pipeline
- **A1 features** are computed once per mission; they are fixed and not data-dependent. The checks print:
  - the slot-6 all-zero fraction, which is asserted to be 100%;
  - the median feature level per channel;
  - each channel's spectral coverage in bins 1–16;
  - the 80-feature mean spectrum.
- **One windowing pass per split** uses `slice_windows_CORRECTED_ALIGNMENT`'s grid, sensor slicing and audio
  interpolation, with the A1 and A2 features side by side. A2's bins come from that split's training missions. On the
  standard split this pass is **checked bit for bit against v24's windows**.
- **`ArmDataset`** returns only its arm's tensor. S1 and S2 take their statistics from the training windows.
- **`UnimodalMamba`** has v24's layers for one modality: encoder, then modality embedding and LayerNorm, then 2 Mamba
  blocks, attention pooling and the classifier. `check_independence` prints the parameter count and module names. It
  asserts there are no LACA, time-embedding, HTT, auxiliary-head, SEP or other-modality parameters, and that the loss is
  class-weighted CE only.
- **Training** uses v24's optimizer, schedule, clipping, epochs, batch size and checkpoint selection, with CE only.

In [ ]:
# ==============================================================================
# UNIMODAL PIPELINE — A1/A2 audio features, one shared windowing pass per split, per-arm datasets (own modality only),
# UnimodalMamba (no LACA, no time embedding, no auxiliary heads, no other-modality parameters), CE-only training.
# v27's fused cells above are NOT modified; the fused model is only the Stage-0 reference.
# ==============================================================================
def _ordered(ms):
    return sorted(ms, key=lambda m: (LABEL[m], m.name))


# ------------------------------------------------------------------------------
# A1 per-buffer feature function (fixed; no data-dependent selection)
# ------------------------------------------------------------------------------
def a1_buffer_spectra(chunks):
    """(n, 1536) raw buffers -> per-channel |rFFT| (n, 5, 129) of the 5 kept, mean-removed channels, and the slot-6 flags.
    Interleaving: sample i belongs to frame i // 6, slot i % 6 (row-major reshape to (256, 6))."""
    chunks = np.asarray(chunks)
    assert chunks.ndim == 2 and chunks.shape[1] == A1_BUFFER_LEN, f"audio buffers {chunks.shape}, expected (n, {A1_BUFFER_LEN})"
    frames = chunks.reshape(len(chunks), A1_FRAMES, A1_SLOTS)
    slot6_zero = np.all(frames[:, :, A1_SLOTS - 1] == 0, axis=1)
    x = frames[:, :, :A1_KEEP_SLOTS].astype(np.float64)
    x = x - x.mean(axis=1, keepdims=True)                              # subtract each channel's mean (per buffer)
    mag = np.abs(np.fft.rfft(x, axis=1))                               # (n, 129, 5): 256-point rFFT per channel
    return np.transpose(mag, (0, 2, 1)), slot6_zero


# A1 features per mission: sqrt(|rFFT|) of bins 1..16 for each of the 5 channels, channel-major (n_chunks, 80), no
# normalization; plus the mission-mean log power (80) for the descriptive analysis. Every buffer's slot 6 is asserted zero.
A1_FEATS, A1_LOGPOW = {}, {}
_n_buf, _n_zero6, _cov_num, _cov_den, _feat_sum, _feat_n, _feat_samples = 0, 0, np.zeros(A1_KEEP_SLOTS), np.zeros(A1_KEEP_SLOTS), 0.0, 0, []
for m in tqdm(ALL_MISSIONS, desc="A1 features", leave=False):
    mag, z6 = a1_buffer_spectra(CORRECTED_PARSED[m]["audio_chunks"])
    _n_buf += len(z6)
    _n_zero6 += int(z6.sum())
    _cov_num += mag[:, :, A1_BINS].sum(axis=(0, 2))
    _cov_den += mag[:, :, 1:].sum(axis=(0, 2))
    A1_LOGPOW[m.name] = np.log(mag[:, :, A1_BINS] ** 2 + 1e-12).mean(axis=0).reshape(-1)
    feats = np.sqrt(mag[:, :, A1_BINS]).reshape(len(mag), -1).astype(np.float32)
    A1_FEATS[m.name] = feats
    _feat_sum = _feat_sum + feats.sum(axis=0)
    _feat_n += len(feats)
    _feat_samples.append(feats[:: max(1, len(feats) // 200)])
print("█" * 110)
print(f"█  A1 CHECK: slot 6 all-zero in {_n_zero6}/{_n_buf} buffers = {100.0 * _n_zero6 / _n_buf:.3f}% (expected 100%)")
assert _n_zero6 == _n_buf, "A1: slot 6 is NOT always zero — the 6-slot interleaving assumption is wrong; stopping"
_samp = np.concatenate(_feat_samples).reshape(-1, A1_KEEP_SLOTS, len(A1_BINS))
print("█  A1 median feature level per channel (sqrt|rFFT|, bins 1-16): "
      + ", ".join(f"ch{c + 1} {np.median(_samp[:, c, :]):.3f}" for c in range(A1_KEEP_SLOTS)))
print("█  A1 spectrum coverage per channel: share of |rFFT| magnitude (bins 1-128) that falls in the kept bins 1-16: "
      + ", ".join(f"ch{c + 1} {100.0 * _cov_num[c] / _cov_den[c]:.1f}%" for c in range(A1_KEEP_SLOTS)))
_mean_spec = (_feat_sum / _feat_n).reshape(A1_KEEP_SLOTS, len(A1_BINS))
print("█  A1 80-feature mean spectrum (rows = channels 1-5, columns = bins 1-16), all missions:")
for c in range(A1_KEEP_SLOTS):
    print("█    ch" + str(c + 1) + " " + " ".join(f"{v:7.2f}" for v in _mean_spec[c]))
print("█" * 110)
del _feat_samples, _samp


# ------------------------------------------------------------------------------
# ONE windowing pass per split: slice_windows_CORRECTED_ALIGNMENT's loop, with several audio feature sets at once
# ------------------------------------------------------------------------------
def slice_windows_multi(mission, label, feats_by_arm, name):
    """slice_windows_CORRECTED_ALIGNMENT's grid, sensor slicing and audio interpolation (Ta=64 real chunk times, corrected
    timestamps), applied to each per-buffer feature set in `feats_by_arm`. Each window carries an id and its mission/day."""
    s_times, telemetry = mission["sensor_times"], mission["telemetry"]
    a_times = mission["audio_times"]
    max_time = min(s_times[-1], a_times[-1])
    windows, i = [], 0
    t_start = max(0.0, float(a_times[0])) if CORRECTED_WINDOW_START == "audio_onset" else 0.0
    while t_start + WINDOW_SEC <= max_time:
        t_end = t_start + WINDOW_SEC
        s_mask = (s_times >= t_start) & (s_times <= t_end)
        if s_mask.sum() < MIN_SENSOR_SAMPLES_PER_WINDOW:
            t_start += STEP_SEC
            continue
        s_idx = np.where(s_mask)[0]
        tau_s_abs = s_times[s_idx[np.linspace(0, len(s_idx) - 1, TS_SENSOR).round().astype(int)]]
        x_s = np.zeros((TS_SENSOR, N_SENSOR_CHANNELS), dtype=np.float32)
        for c in range(N_SENSOR_CHANNELS):
            x_s[:, c] = np.interp(tau_s_abs, s_times[s_mask], telemetry[s_mask, c])
        a_mask = (a_times >= t_start) & (a_times <= t_end)
        if a_mask.sum() >= 2:
            a_idx = np.where(a_mask)[0]
            tau_a_abs = a_times[a_idx[np.linspace(0, len(a_idx) - 1, TA_AUDIO).round().astype(int)]]
        else:
            tau_a_abs = np.linspace(t_start, t_end, TA_AUDIO)
        x_a = {}
        for arm, feats in feats_by_arm.items():
            xa = np.zeros((TA_AUDIO, feats.shape[1]), dtype=np.float32)
            for c in range(feats.shape[1]):
                xa[:, c] = np.interp(tau_a_abs, a_times, feats[:, c], left=0.0, right=0.0)
            x_a[arm] = xa
        windows.append({"id": f"{name}#{i:04d}", "mission": name, "day": mission_day(name), "label": int(label),
                        "t_start": t_start, "x_s": x_s, "x_a": x_a})
        i += 1
        t_start += STEP_SEC
    return windows


def build_base(split_missions, name):
    """All windows of one split (train/val/test missions) with A1 and A2 audio and raw sensor. A2's strongest-64 bins come
    from THIS split's training missions only."""
    bins = select_strongest_bins([CORRECTED_PARSED[m] for m in _ordered(split_missions["train"])], N_AUDIO_BINS)
    splits = {}
    for s in ("train", "val", "test"):
        splits[s] = []
        for m in _ordered(split_missions[s]):
            raw = CORRECTED_PARSED[m]
            w = slice_windows_multi(raw, LABEL[m], {"A1": A1_FEATS[m.name], "A2": audio_features(raw["audio_chunks"], bins)}, m.name)
            assert len(w) == WINDOWS_PER_MISSION[m.name], f"{m.name}: {len(w)} windows vs {WINDOWS_PER_MISSION[m.name]}"
            splits[s].extend(w)
    for s in ("val", "test"):
        present = {w["label"] for w in splits[s]}
        assert present == set(range(N_CLASSES)), f"{name} {s}: classes {sorted(present)} — metrics need all 5"
    return {"name": name, "bins": bins, "splits": splits,
            "n_missions": {s: len(split_missions[s]) for s in ("train", "val", "test")}}


STANDARD_SPLIT = {s: [m for lab in sorted(corrected_splits[s]) for m in corrected_splits[s][lab]] for s in ("train", "val", "test")}
# --- check: the shared windowing pass reproduces v24's CORRECTED windows exactly on the standard split (A2 == v24 audio) ---
_std = build_base(STANDARD_SPLIT, "standard")
_by_m = defaultdict(list)
for s in ("train", "val", "test"):
    for w, n in zip(CORRECTED_ALIGNMENT["split_windows"][s], CORRECTED_ALIGNMENT["window_missions"][s]):
        _by_m[n].append(w)
_new_by_m = defaultdict(list)
for s in ("train", "val", "test"):
    for w in _std["splits"][s]:
        _new_by_m[w["mission"]].append(w)
WINDOWING_MATCHES_V24 = (_std["bins"].tolist() == CORRECTED_ALIGNMENT["bins"].tolist() and set(_by_m) == set(_new_by_m) and all(
    len(_by_m[n]) == len(_new_by_m[n]) and all(np.array_equal(a["x_s"], b["x_s"]) and np.array_equal(a["x_a"], b["x_a"]["A2"])
                                               for a, b in zip(_by_m[n], _new_by_m[n])) for n in _by_m))
print(f"[WINDOWING] standard split: A2 audio + raw sensor of every window == v24's CORRECTED windows (bit for bit): "
      + ("✅" if WINDOWING_MATCHES_V24 else "🚨 MISMATCH"))
assert WINDOWING_MATCHES_V24, "the shared windowing pass does not reproduce v24's windows"
BASE_CACHE = {("standard", 0): _std}
del _by_m, _new_by_m


# ------------------------------------------------------------------------------
# Per-arm datasets: ONLY that arm's modality is returned
# ------------------------------------------------------------------------------
class ArmDataset(Dataset):
    """A1/A2: x = the arm's audio features (Ta=64 x dim), no normalization. S1: x = (x_s - train mean) / train std.
    S2: x = (x_s - its own window mean) / std of the mean-removed TRAINING values. Stats come from the training windows."""

    def __init__(self, windows, arm, stats=None):
        self.windows, self.arm, self.modality = windows, arm, ARM_INFO[arm][0]
        if self.modality == "sensor" and stats is None:            # training split: compute the stats here
            xs = np.stack([w["x_s"] for w in windows])
            if arm == "S1":
                all_s = xs.reshape(-1, N_SENSOR_CHANNELS)
                stats = (all_s.mean(axis=0, keepdims=True), all_s.std(axis=0, keepdims=True) + SENSOR_STD_EPS)
            else:
                centered = (xs - xs.mean(axis=1, keepdims=True)).reshape(-1, N_SENSOR_CHANNELS)
                stats = (None, centered.std(axis=0, keepdims=True) + SENSOR_STD_EPS)
        self.stats = stats
        self.ids = [w["id"] for w in windows]

    def __len__(self):
        return len(self.windows)

    def x(self, w):
        if self.modality == "audio":
            return w["x_a"][self.arm]
        if self.arm == "S1":
            return (w["x_s"] - self.stats[0]) / self.stats[1]
        return (w["x_s"] - w["x_s"].mean(axis=0, keepdims=True)) / self.stats[1]

    def __getitem__(self, idx):
        w = self.windows[idx]
        return {"x": torch.tensor(self.x(w), dtype=torch.float32), "label": torch.tensor(w["label"], dtype=torch.long)}


def ids_sha1(ids):
    return hashlib.sha1("\n".join(ids).encode()).hexdigest()[:16]


def make_arm_bundle(base, arm):
    tr = ArmDataset(base["splits"]["train"], arm)
    va = ArmDataset(base["splits"]["val"], arm, tr.stats)
    te = ArmDataset(base["splits"]["test"], arm, tr.stats)
    test_w = base["splits"]["test"]
    return {"arm": arm, "train_ds": tr, "val_ds": va, "test_ds": te,
            "train_loader": DataLoader(tr, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True, num_workers=NUM_WORKERS),
            "val_loader": DataLoader(va, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True, num_workers=NUM_WORKERS),
            "test_loader": DataLoader(te, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True, num_workers=NUM_WORKERS),
            "class_weights": train_class_weights(base["splits"]["train"]).to(device),   # this split's TRAINING windows
            "test_ids": np.array(te.ids), "test_missions": np.array([w["mission"] for w in test_w]),
            "test_days": np.array([w["day"] for w in test_w]), "test_labels": np.array([w["label"] for w in test_w], dtype=np.int64),
            "ids_sha1": {s: ids_sha1(d.ids) for s, d in (("train", tr), ("val", va), ("test", te))}}


def s2_input_check(bundle):
    """S2 hook: mean |window mean| per channel of the raw sensor input vs of S2's model input (expected ~0 after)."""
    ws = bundle["train_ds"].windows
    before = np.mean([np.abs(w["x_s"].mean(axis=0)) for w in ws], axis=0)
    after = np.mean([np.abs(bundle["train_ds"].x(w).mean(axis=0)) for w in ws], axis=0)
    print(f"[S2 CHECK] mean |window mean| per channel (training windows) before removal "
          f"{np.round(before, 4).tolist()} -> after {np.round(after, 8).tolist()} (expected ~0)")
    return float(after.max())


# ------------------------------------------------------------------------------
# UnimodalMamba: encoder -> + modality embedding -> LayerNorm -> 2 Mamba blocks -> attention pooling -> classifier
# ------------------------------------------------------------------------------
class UnimodalMamba(nn.Module):
    """The fused model's single-modality path, with v24's layer shapes: the same encoder for this modality, a modality
    embedding + LayerNorm (HTT with the time embedding OFF, one modality), v24's Mamba blocks, attention pooling and
    classifier head. No LACA, no time embedding, no SEP token, no auxiliary heads, no other-modality parameters."""

    def __init__(self, arm):
        super().__init__()
        self.arm, (self.modality, _, in_dim) = arm, ARM_INFO[arm]
        if self.modality == "audio":
            self.audio_encoder = nn.Sequential(nn.Linear(in_dim, D_MODEL), nn.LayerNorm(D_MODEL), nn.GELU(), nn.Linear(D_MODEL, D_MODEL))
        else:
            self.sensor_encoder = nn.Sequential(
                nn.Conv1d(N_SENSOR_CHANNELS, 64, kernel_size=5, padding=2), nn.BatchNorm1d(64), nn.GELU(),
                nn.Conv1d(64, D_MODEL, kernel_size=3, padding=1), nn.BatchNorm1d(D_MODEL), nn.GELU())
        self.modality_embedding = nn.Embedding(1, D_MODEL)
        self.token_norm = nn.LayerNorm(D_MODEL)
        self.mamba_blocks = build_fusion_blocks(D_MODEL, MAMBA_NUM_BLOCKS, MAMBA_D_STATE, MAMBA_D_CONV)
        self.pool_attn = nn.Sequential(nn.Linear(D_MODEL, 64), nn.Tanh(), nn.Linear(64, 1))
        self.classifier = nn.Sequential(nn.Dropout(DROPOUT), nn.Linear(D_MODEL, D_MODEL // 2), nn.GELU(),
                                        nn.Linear(D_MODEL // 2, N_CLASSES))

    def forward(self, x):
        z = self.audio_encoder(x) if self.modality == "audio" else self.sensor_encoder(x.transpose(1, 2)).transpose(1, 2)
        h = self.token_norm(z + self.modality_embedding.weight.view(1, 1, -1))
        for block in self.mamba_blocks:
            h = block(h)
        weights = F.softmax(self.pool_attn(h), dim=1)
        return self.classifier((h * weights).sum(dim=1))


def check_independence(model, criterion, verbose):
    """Assert: no LACA / time embedding / HTT / auxiliary heads / SEP, no parameters of the other modality, CE-only loss."""
    names = [n for n, _ in model.named_parameters()]
    other = "sensor" if model.modality == "audio" else "audio"
    forbidden = ["laca", "lag", "time_embedding", "htt", "aux", "sep_token", other]
    bad = [n for n in names if any(f in n.lower() for f in forbidden)]
    assert not bad, f"{model.arm}: forbidden parameters {bad}"
    assert type(criterion) is nn.CrossEntropyLoss and criterion.weight is not None, "loss must be class-weighted CE only"
    mamba = sum(p.numel() for p in model.mamba_blocks.parameters())
    if MAMBA_CLASS is not None:
        assert mamba == EXPECTED_MAMBA_PARAMS, f"Mamba params {mamba:,} != {EXPECTED_MAMBA_PARAMS:,}"
    n_params = sum(p.numel() for p in model.parameters())
    if verbose:
        print(f"[INDEPENDENCE {model.arm}] {n_params:,} params | modules {[n for n, _ in model.named_children()]} | Mamba "
              f"{mamba:,} ({type(model.mamba_blocks[0]).__name__}) | no LACA / time embedding / HTT / aux heads / SEP / "
              f"{other} parameters ✅ | loss = class-weighted CrossEntropyLoss only ✅")
    return n_params


def predict_arm(model, loader):
    model.eval()
    probs, labels = [], []
    with torch.no_grad():
        for batch in loader:
            probs.append(F.softmax(model(batch["x"].to(device)), dim=-1).cpu().numpy())
            labels.append(batch["label"].numpy())
    return np.concatenate(probs), np.concatenate(labels)


INDEPENDENCE_PRINTED = set()


def train_unimodal(arm, seed, bundle, run_name):
    """v24's optimization (AdamW, cosine, grad clip, epochs, batch, best validation macro-F1) with a CE-only loss."""
    t0 = time.time()
    set_all_seeds(seed)
    model = UnimodalMamba(arm).to(device)
    criterion = nn.CrossEntropyLoss(weight=bundle["class_weights"])
    n_params = check_independence(model, criterion, verbose=arm not in INDEPENDENCE_PRINTED)
    INDEPENDENCE_PRINTED.add(arm)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR_MIN)
    best_val_f1, best_state, best_epoch = 0.0, None, None
    for epoch in range(1, EPOCHS + 1):
        model.train()
        running = 0.0
        for batch in tqdm(bundle["train_loader"], desc=f"{run_name} epoch {epoch:02d}/{EPOCHS}", leave=False):
            x, y = batch["x"].to(device, non_blocking=True), batch["label"].to(device, non_blocking=True)
            optimizer.zero_grad()
            loss = criterion(model(x), y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP_NORM)
            optimizer.step()
            running += loss.item()
        scheduler.step()
        val_probs, val_labels = predict_arm(model, bundle["val_loader"])
        val_f1 = compute_metrics(val_labels, val_probs)["macro_f1"]
        improved = val_f1 > best_val_f1
        if improved:
            best_val_f1, best_state, best_epoch = val_f1, copy.deepcopy(model.state_dict()), epoch
        print(f"[{run_name}] epoch {epoch:02d}/{EPOCHS} | train CE {running / len(bundle['train_loader']):.4f} | val macro-F1 "
              f"{val_f1:.4f}{'  * best' if improved else ''}")
    model.load_state_dict(best_state)
    test_probs, test_labels = predict_arm(model, bundle["test_loader"])
    assert np.array_equal(test_labels, bundle["test_labels"])
    ev = evaluate_probs(test_labels, test_probs)
    torch.save(best_state, CHECKPOINT_DIR / f"{run_name}.pt")
    probs_file = PROBS_DIR / f"{run_name}.npz"
    np.savez(probs_file, window_ids=bundle["test_ids"], mission_ids=bundle["test_missions"], day=bundle["test_days"],
             y_true=test_labels, probs=test_probs.astype(np.float32))
    return model, {**ev, **per_class_prf(test_labels, test_probs), "n_params": n_params, "best_epoch": best_epoch,
                   "best_val_macro_f1": round(best_val_f1, 6), "train_seconds": round(time.time() - t0, 1),
                   "probs_file": str(probs_file)}

## 9. Runs — Stage A (day-out), Stage B (standard), Stage C (optional)
- **Order:** seed-major (seed → fold → arm), so an interrupted session still leaves complete arm sets.
- **Window ids:** checked identical across arms for each fold before training.
- **S2 check:** once per fold, the mean |window mean| before and after removal.
- **Progress:** the first completed run of each arm is printed in full.
- **Logging:** each run is written to the CSV immediately, rows already there are skipped, and GPU memory is freed after
  every run.

In [ ]:
# ==============================================================================
# DRIVER — Stage A (day-out), Stage B (standard split), Stage C (matched-random, only if RUN_MATCHED_RANDOM).
# Seed-major order (a partial session still leaves complete arm sets); one CSV row per finished run; runs already in the
# CSV are skipped; GPU freed after every run.
# ==============================================================================
STAGES = [("A", "day_out", list(range(len(DAYS))), SEEDS), ("B", "standard", [0], SEEDS_STANDARD)]
if RUN_MATCHED_RANDOM:
    STAGES.append(("C", "matched_random", list(range(len(DAYS))), SEEDS))
PLANNED = [(stage, arm, p, k, s) for stage, p, folds, seeds in STAGES for s in seeds for k in folds for arm in ARMS]
print(f"[PLAN] {len(PLANNED)} unimodal runs: " + ", ".join(f"Stage {st} {p} {len(f) * len(sd) * len(ARMS)}"
                                                           for st, p, f, sd in STAGES)
      + ("" if RUN_MATCHED_RANDOM else " | Stage C matched-random is OFF (RUN_MATCHED_RANDOM = False)"))


def split_of(protocol, k):
    if protocol == "standard":
        return STANDARD_SPLIT, "original v24 split"
    f = FOLDS[(protocol, k)]
    return {s: f[s] for s in ("train", "val", "test")}, f["fold_day"]


WINDOW_ID_SHA1 = {}      # (protocol, fold) -> {arm: {split: sha1}} — must be identical across arms


def get_base(protocol, k):
    if (protocol, k) not in BASE_CACHE:
        split, _ = split_of(protocol, k)
        BASE_CACHE[(protocol, k)] = build_base(split, f"{protocol}_{k}")
    base = BASE_CACHE[(protocol, k)]
    if (protocol, k) not in WINDOW_ID_SHA1:     # window-id hook: every arm's datasets must list the SAME windows
        per_arm = {arm: {s: ids_sha1([w["id"] for w in ArmDataset(base["splits"][s], arm, stats=(0.0, 1.0)).windows])
                         for s in ("train", "val", "test")} for arm in ARMS}
        WINDOW_ID_SHA1[(protocol, k)] = per_arm
        assert all(v == per_arm[ARMS[0]] for v in per_arm.values()), f"{protocol} fold {k}: window ids differ across arms"
        print(f"[WINDOW IDS] {protocol} fold {k}: identical across {ARMS} ✅ | test sha1 {per_arm[ARMS[0]]['test']} | "
              f"windows train/val/test {'/'.join(str(len(base['splits'][s])) for s in ('train', 'val', 'test'))}")
    return base


FIRST_RUN_SHOWN = set()   # arms whose first completed run this session was printed in full
S2_CHECK = {}             # (protocol, fold) -> max over channels of S2's mean |window mean| after removal (expected ~0)
t_drv, n_new = time.time(), 0
for stage, protocol, folds, seeds in STAGES:
    print("\n" + "#" * 110 + f"\n#  STAGE {stage} — {protocol} | seeds {seeds} x folds {folds} x arms {ARMS}\n" + "#" * 110)
    for s in seeds:
        for k in folds:
            done = done_keys()
            todo = [arm for arm in ARMS if (arm, protocol, k, s) not in done]
            if not todo:
                print(f"[SKIP] {protocol} fold {k} seed {s}: all arms already in {RESULTS_CSV.name}")
                continue
            base = get_base(protocol, k)
            _, fold_day = split_of(protocol, k)
            for arm in todo:
                run_name = f"{arm}_{protocol}_{k}_{s}"
                print("\n" + "=" * 110 + f"\n{run_name} — {ARM_INFO[arm][1]} | fold {k} ({fold_day})\n" + "=" * 110)
                bundle = make_arm_bundle(base, arm)
                assert bundle["ids_sha1"] == WINDOW_ID_SHA1[(protocol, k)][arm]
                if arm == "S2" and (protocol, k) not in S2_CHECK:
                    S2_CHECK[(protocol, k)] = s2_input_check(bundle)
                model, result = train_unimodal(arm, s, bundle, run_name)
                log_row({"stage": stage, "arm": arm, "protocol": protocol, "fold": k, "fold_day": fold_day, "seed": s,
                         **{m: result[m] for m in METRICS}, **{k2: v for k2, v in result.items() if k2[:-1] in ("precision_c", "recall_c", "f1_c")},
                         "confusion_matrix": json.dumps(np.asarray(result["confusion_matrix"]).tolist()),
                         "n_test_windows": len(bundle["test_ds"]), "n_test_missions": base["n_missions"]["test"],
                         "n_train_windows": len(bundle["train_ds"]), "n_val_windows": len(bundle["val_ds"]),
                         "window_ids_sha1": bundle["ids_sha1"]["test"], "n_params": result["n_params"],
                         "best_epoch": result["best_epoch"], "best_val_macro_f1": result["best_val_macro_f1"],
                         "train_seconds": result["train_seconds"], "probs_file": result["probs_file"]})
                n_new += 1
                if arm not in FIRST_RUN_SHOWN:
                    FIRST_RUN_SHOWN.add(arm)
                    print("█" * 110 + f"\n█  FIRST COMPLETED {arm} RUN THIS SESSION ({run_name}): accuracy {result['accuracy']:.4f} | "
                          f"macro-F1 {result['macro_f1']:.4f} | balanced accuracy {result['balanced_accuracy']:.4f} | macro-AUROC "
                          f"{result['macro_auroc']:.4f}\n█  recall per class " + ", ".join(
                              f"C{c} {result[f'recall_c{c}']:.3f}" for c in range(N_CLASSES)) + "\n" + "█" * 110)
                    print_confusion(result, run_name)
                else:
                    print(f"[{run_name}] macro-F1 {result['macro_f1']:.4f} | balanced accuracy {result['balanced_accuracy']:.4f} | "
                          f"recall " + ", ".join(f"C{c} {result[f'recall_c{c}']:.3f}" for c in range(N_CLASSES)))
                del model, result, bundle
                free_gpu()
                n_done = sum(1 for key in done_keys() if key[0] in ARMS)
                el = time.time() - t_drv
                print(f">>> {n_done}/{len(PLANNED)} unimodal runs in the CSV ({n_new} this session) | elapsed {el / 60:.1f} min | "
                      f"est. remaining {el / n_new * max(len(PLANNED) - n_done, 0) / 60:.1f} min")
    for key in [key for key in BASE_CACHE if key[0] == protocol]:
        del BASE_CACHE[key]
    gc.collect()

_rows = [r for r in load_rows() if r["arm"] in ARMS]
_have = {(r["arm"], r["protocol"], int(r["fold"]), int(r["seed"])) for r in _rows}
MISSING = [key for key in PLANNED if key[1:] not in _have]
print(f"\nUnimodal runs: {len(PLANNED) - len(MISSING)}/{len(PLANNED)} planned runs in {RESULTS_CSV.name} ({n_new} trained this "
      f"session, {(time.time() - t_drv) / 60:.1f} min)" + (f" | MISSING {len(MISSING)}: the analysis uses what has completed"
                                                          if MISSING else " ✅"))

## 10. Analysis 1–5, 7
- **Pooled out-of-fold day-out results.** Per class P/R/F1 per arm, seed-averaged, with **95% mission-level cluster
  bootstrap CIs**: missions are resampled within class 2000 times, never windows. Also per-fold macro-F1 and balanced
  accuracy.
- **Characterization card**, class × arm:
  - STRONG if the recall CI's lower bound is ≥ 0.70;
  - WEAK if its upper bound is < 0.70;
  - UNCERTAIN otherwise.
- **Contrasts:** A1 vs A2 and S1 vs S2, paired by fold, each with what it can and cannot separate.
- **Complementarity:** A1 × S1 and A1 × S2 on identical windows, at window and mission level, including the oracle
  ceiling, per-class rescues and probability averaging. The averaging is **analysis only, not a proposed model**.
- **Fused reference.** The first available source is used:
  - v27's per-window probabilities (paired analysis possible);
  - otherwise v27's CSV;
  - otherwise the hardcoded v27 numbers.

  v24's standard-split fused scores are listed alongside.
- **Stage C gap** per arm, if it was run.

In [ ]:
# ==============================================================================
# ANALYSIS 1-5, 7 — read back from the CSV + per-run probability files; uses whatever has completed
# ==============================================================================
from scipy import stats

ROWS = {(r["arm"], r["protocol"], int(r["fold"]), int(r["seed"])): r for r in load_rows() if r["arm"] in ARMS}
PRED = {}
for key in ROWS:
    path = PROBS_DIR / f"{key[0]}_{key[1]}_{key[2]}_{key[3]}.npz"
    if path.exists():
        with np.load(path) as z:
            PRED[key] = {k: z[k] for k in ("window_ids", "mission_ids", "day", "y_true", "probs")}
    else:
        print(f"⚠️ {path.name} missing (row in CSV but no probability file): left out of the window/mission-level analyses")
DAYOUT_FOLDS = list(range(len(DAYS)))
LABELS_C = [f"C{c}" for c in range(N_CLASSES)]


def prf_from_cm(cm):
    cm = np.asarray(cm, dtype=float)
    tp = np.diag(cm)
    recall = tp / np.maximum(cm.sum(axis=1), 1e-12)
    precision = np.where(cm.sum(axis=0) > 0, tp / np.maximum(cm.sum(axis=0), 1e-12), 0.0)
    f1 = np.where(precision + recall > 0, 2 * precision * recall / np.maximum(precision + recall, 1e-12), 0.0)
    return {"precision": precision, "recall": recall, "f1": f1, "macro_f1": float(f1.mean())}


def complete_seeds(arm, protocol="day_out", folds=DAYOUT_FOLDS, seeds=SEEDS):
    return [s for s in seeds if all((arm, protocol, k, s) in PRED for k in folds)]


def mission_counts(arm, seeds, protocol="day_out", folds=DAYOUT_FOLDS):
    """Missions (each tested once per seed across the folds), their class, and per-seed predicted-class window counts."""
    per_seed = {}
    for s in seeds:
        mids = np.concatenate([PRED[(arm, protocol, k, s)]["mission_ids"] for k in folds])
        y = np.concatenate([PRED[(arm, protocol, k, s)]["y_true"] for k in folds])
        pred = np.concatenate([PRED[(arm, protocol, k, s)]["probs"].argmax(axis=1) for k in folds])
        per_seed[s] = (mids, y, pred)
    missions = sorted(set(per_seed[seeds[0]][0]))
    assert all(sorted(set(v[0])) == missions for v in per_seed.values()), f"{arm}: seeds cover different missions"
    index = {m: i for i, m in enumerate(missions)}
    C = np.zeros((len(missions), len(seeds), N_CLASSES))
    ym = np.zeros(len(missions), dtype=int)
    for j, s in enumerate(seeds):
        mids, y, pred = per_seed[s]
        np.add.at(C, (np.array([index[m] for m in mids]), j, pred), 1)
        ym[[index[m] for m in mids]] = y
    return missions, ym, C


def pooled_with_ci(arm, seeds):
    """Seed-averaged pooled per-class P/R/F1 + 95% mission-level cluster bootstrap CIs (missions resampled within class)."""
    missions, ym, C = mission_counts(arm, seeds)
    cls_idx = [np.where(ym == c)[0] for c in range(N_CLASSES)]

    def stats_for(weights):
        out = []
        for j in range(len(seeds)):
            cm = np.stack([weights[cls_idx[c]] @ C[cls_idx[c], j, :] for c in range(N_CLASSES)])
            out.append(prf_from_cm(cm))
        return {k: np.mean([o[k] for o in out], axis=0) for k in ("precision", "recall", "f1", "macro_f1")}

    point = stats_for(np.ones(len(missions)))
    rng = np.random.default_rng(BOOTSTRAP_SEED)                     # same resamples for every arm (common random numbers)
    boot = defaultdict(list)
    for _ in range(BOOTSTRAP_RESAMPLES):
        w = np.zeros(len(missions))
        for idx in cls_idx:
            np.add.at(w, idx[rng.integers(0, len(idx), len(idx))], 1)
        b = stats_for(w)
        for k in ("precision", "recall", "f1", "macro_f1"):
            boot[k].append(b[k])
    ci = {k: (np.percentile(np.array(v), 2.5, axis=0), np.percentile(np.array(v), 97.5, axis=0)) for k, v in boot.items()}
    return {"point": point, "ci": ci, "n_missions": len(missions), "missions_per_class": [len(i) for i in cls_idx],
            "seeds": seeds, "cm_sum": sum(np.stack([np.stack([C[cls_idx[c], j, :].sum(axis=0) for c in range(N_CLASSES)])
                                                    for j in range(len(seeds))]))}


# --- 5 (loaded first so the fused reference can join the tables): fused Condition D day-out reference ---
FUSED_MODE, FUSED_POOLED = None, None
_v27 = {}
for k in DAYOUT_FOLDS:
    for s in SEEDS:
        p = sorted(glob.glob(V27_PROBS_GLOB.format(name=f"day_out_fold{k}_D_seed{s}_test_probs.npy"), recursive=True))
        l = sorted(glob.glob(V27_PROBS_GLOB.format(name=f"day_out_fold{k}_D_seed{s}_test_labels.npy"), recursive=True))
        if p and l:
            _v27[(k, s)] = (p[0], l[0])
if len(_v27) == len(DAYOUT_FOLDS) * len(SEEDS):
    for (k, s), (pp, lp) in _v27.items():
        test = _ordered(FOLDS[("day_out", k)]["test"])      # v27's window order: missions by (class, name), windows in time order
        ids = np.array([f"{m.name}#{i:04d}" for m in test for i in range(WINDOWS_PER_MISSION[m.name])])
        mids = np.array([m.name for m in test for _ in range(WINDOWS_PER_MISSION[m.name])])
        y = np.load(lp)
        assert np.array_equal(y, np.array([LABEL[m] for m in test for _ in range(WINDOWS_PER_MISSION[m.name])])), \
            f"v27 fold {k} seed {s}: labels do not match this notebook's fold windows"
        PRED[("FUSED_v27", "day_out", k, s)] = {"window_ids": ids, "mission_ids": mids, "day": np.array([mission_day(n) for n in mids]),
                                               "y_true": y, "probs": np.load(pp)}
    FUSED_MODE = "v27 per-window probabilities (attached)"
else:
    _csv = sorted(glob.glob(V27_CSV_GLOB, recursive=True))
    if _csv:
        with open(_csv[0], newline="") as f:
            _r = [r for r in csv.DictReader(f) if r["protocol"] == "day_out"]
        _per_seed = [prf_from_cm(sum(np.array(json.loads(r["confusion_matrix"])) for r in _r if int(r["seed"]) == s)) for s in SEEDS]
        FUSED_POOLED = {"macro_f1": float(np.mean([p["macro_f1"] for p in _per_seed])),
                        "recall": np.mean([p["recall"] for p in _per_seed], axis=0),
                        "precision": np.mean([p["precision"] for p in _per_seed], axis=0),
                        "f1": np.mean([p["f1"] for p in _per_seed], axis=0)}
        FUSED_MODE = f"v27 results CSV ({_csv[0]}): pooled confusion matrices, no mission-level CI, no paired analysis"
    else:
        FUSED_POOLED = {"macro_f1": V27_FUSED_DAYOUT["macro_f1"], "recall": np.array(V27_FUSED_DAYOUT["recall"]),
                        "precision": None, "f1": None}
        FUSED_MODE = "HARDCODED v27 printed pooled day-out numbers (no v27 files attached): no CI, no paired analysis"
print(f"[FUSED REFERENCE] {FUSED_MODE}")

# --- 1. pooled out-of-fold day-out per class, with mission-level cluster bootstrap CIs ---
POOLED = {}
for arm in ARMS + (["FUSED_v27"] if FUSED_MODE and FUSED_MODE.startswith("v27 per-window") else []):
    seeds = complete_seeds(arm)
    if not seeds:
        print(f"⚠️ {arm}: no seed has all {len(DAYOUT_FOLDS)} day-out folds complete — left out of the pooled analysis")
        continue
    POOLED[arm] = pooled_with_ci(arm, seeds)
    P = POOLED[arm]
    print("\n" + "=" * 118 + f"\nPOOLED OUT-OF-FOLD DAY-OUT — {arm} ({ARM_INFO[arm][1] if arm in ARM_INFO else 'fused Condition D, v27'}) | "
          f"seeds {seeds} averaged | {P['n_missions']} missions {P['missions_per_class']} | 95% CI: {BOOTSTRAP_RESAMPLES} "
          f"mission-level resamples within class\n" + "=" * 118)
    print(f"    {'class':<7}{'precision [95% CI]':<28}{'recall [95% CI]':<28}{'F1 [95% CI]':<28}")
    for c in range(N_CLASSES):
        cell = lambda k: f"{P['point'][k][c]:.3f} [{P['ci'][k][0][c]:.3f}, {P['ci'][k][1][c]:.3f}]"
        print(f"    {LABELS_C[c]:<7}{cell('precision'):<28}{cell('recall'):<28}{cell('f1'):<28}")
    print(f"    macro-F1 {P['point']['macro_f1']:.4f} [{P['ci']['macro_f1'][0]:.4f}, {P['ci']['macro_f1'][1]:.4f}]")
if FUSED_POOLED is not None:
    print(f"\n    FUSED Condition D day-out (v27, {FUSED_MODE.split(':')[0]}): macro-F1 {FUSED_POOLED['macro_f1']:.4f} | recall "
          + ", ".join(f"C{c} {FUSED_POOLED['recall'][c]:.4f}" for c in range(N_CLASSES)))

# --- 1b. per-fold (per-day) macro-F1 and balanced accuracy, seed-averaged ---
print("\n" + "=" * 118 + "\nPER-FOLD (PER-DAY) DAY-OUT — macro-F1 / balanced accuracy, mean over the seeds present (do NOT read per-fold "
      "per-class recall: 2-8 missions)\n" + "=" * 118)
print(f"    {'fold':<5}{'day':<13}" + "".join(f"{a:>18}" for a in ARMS))
PER_FOLD = {}
for k in DAYOUT_FOLDS:
    cells = []
    for a in ARMS:
        rs = [ROWS[(a, "day_out", k, s)] for s in SEEDS if (a, "day_out", k, s) in ROWS]
        PER_FOLD[(a, k)] = (np.mean([float(r["macro_f1"]) for r in rs]), np.mean([float(r["balanced_accuracy"]) for r in rs])) if rs else None
        cells.append(f"{PER_FOLD[(a, k)][0]:.3f}/{PER_FOLD[(a, k)][1]:.3f}" if rs else "—")
    print(f"    {k:<5}{DAYS[k]:<13}" + "".join(f"{c:>18}" for c in cells) + ("   <- SMALL fold (10 missions)" if DAYS[k] == SMALL_FOLD_DAY else ""))

# --- 2. characterization card: class x arm ---
CARD = {}
print("\n" + "█" * 118 + f"\n█  CHARACTERIZATION CARD — pooled day-out recall: STRONG if CI low >= {CARD_RECALL}, WEAK if CI high < "
      f"{CARD_RECALL}, else UNCERTAIN\n" + "█" * 118)
_card_arms = list(POOLED)
print("█    " + f"{'class':<7}" + "".join(f"{a:>26}" for a in _card_arms))
for c in range(N_CLASSES):
    cells = []
    for a in _card_arms:
        lo, hi = POOLED[a]["ci"]["recall"][0][c], POOLED[a]["ci"]["recall"][1][c]
        CARD[(a, c)] = "STRONG" if lo >= CARD_RECALL else ("WEAK" if hi < CARD_RECALL else "UNCERTAIN")
        cells.append(f"{CARD[(a, c)]} {POOLED[a]['point']['recall'][c]:.2f} [{lo:.2f},{hi:.2f}]")
    print("█    " + f"{LABELS_C[c]:<7}" + "".join(f"{x:>26}" for x in cells))
print("█" * 118)

# --- 3. representation contrasts, paired by fold ---
CONTRASTS = {}
print("\n" + "=" * 118 + "\nREPRESENTATION CONTRASTS — paired by day-out fold (seed-averaged macro-F1)\n" + "=" * 118)
_explain = {
    ("A1", "A2"): "A1 - A2 changes several things at once: channel handling (5 de-interleaved channels vs one FFT over the interleaved "
                  "stream), frequency band (per-channel bins 1-16 vs the strongest-64 bins, which sit at multiples of ~256 bins), "
                  "feature count (80 vs 64) and fixed vs fold-selected bins. It shows whether the de-interleaved representation "
                  "carries more or less class information for this architecture. It CANNOT say which of these changes is responsible.",
    ("S1", "S2"): "S1 - S2 removes each window's mean: the static offset per axis (gravity direction = attitude, sensor bias) and all "
                  "content slower than ~1 window (< ~0.4 Hz), and rescales by the vibration-only std. A positive value says the "
                  "removed slow/static part helps. It CANNOT separate gravity/attitude from bias from slow manoeuvres, and cannot "
                  "tell whether that part is fault-related or a day/flight-profile fingerprint (see the day-out vs matched gap)."}
for a, b in (("A1", "A2"), ("S1", "S2")):
    diffs = [PER_FOLD[(a, k)][0] - PER_FOLD[(b, k)][0] for k in DAYOUT_FOLDS if PER_FOLD.get((a, k)) and PER_FOLD.get((b, k))]
    if not diffs:
        print(f"    {a} - {b}: no fold has both arms — skipped")
        continue
    d = np.array(diffs)
    CONTRASTS[(a, b)] = {"mean": float(d.mean()), "sd": float(d.std(ddof=1)) if len(d) > 1 else float("nan"), "wins": int((d > 0).sum()), "n": len(d)}
    print(f"    {a} - {b}: per fold " + ", ".join(f"{x:+.3f}" for x in d) + f" | mean {d.mean():+.4f} | SD "
          f"{CONTRASTS[(a, b)]['sd']:.4f} | {a} better in {CONTRASTS[(a, b)]['wins']}/{len(d)} folds")
    if a in POOLED and b in POOLED:
        print(f"      pooled per-class recall change ({a} - {b}): " + ", ".join(
            f"C{c} {POOLED[a]['point']['recall'][c] - POOLED[b]['point']['recall'][c]:+.3f}" for c in range(N_CLASSES)))
    print("      " + textwrap.fill(_explain[(a, b)], 112, subsequent_indent="      "))


# --- 4. complementarity (analysis only): identical windows, window and mission level ---
def complementarity(audio_arm, imu_arm, label):
    keys = [(k, s) for k in DAYOUT_FOLDS for s in SEEDS if (audio_arm, "day_out", k, s) in PRED and (imu_arm, "day_out", k, s) in PRED]
    if not keys:
        print(f"    {label}: no (fold, seed) with both runs — skipped")
        return None
    W = defaultdict(int)
    Mi = defaultdict(int)
    miss_by_class, rescued_by_class = np.zeros(N_CLASSES, int), np.zeros(N_CLASSES, int)
    avg_f1 = {}
    per_seed_cm = {s: np.zeros((N_CLASSES, N_CLASSES)) for s in SEEDS}
    for k, s in keys:
        A, I = PRED[(audio_arm, "day_out", k, s)], PRED[(imu_arm, "day_out", k, s)]
        assert np.array_equal(A["window_ids"], I["window_ids"]), f"fold {k} seed {s}: window ids differ between {audio_arm} and {imu_arm}"
        y = A["y_true"]
        ra, ri = A["probs"].argmax(1) == y, I["probs"].argmax(1) == y
        for name, mask in (("both right", ra & ri), ("audio only right", ra & ~ri), ("IMU only right", ~ra & ri), ("both wrong", ~ra & ~ri)):
            W[name] += int(mask.sum())
        W["n"] += len(y)
        avg = (A["probs"] + I["probs"]) / 2
        np.add.at(per_seed_cm[s], (y, avg.argmax(1)), 1)
        for m in np.unique(A["mission_ids"]):
            sel = A["mission_ids"] == m
            yc = int(y[sel][0])
            pa, pi = A["probs"][sel].mean(0).argmax() == yc, I["probs"][sel].mean(0).argmax() == yc
            for name, v in (("both right", pa and pi), ("audio only right", pa and not pi), ("IMU only right", pi and not pa),
                            ("both wrong", not pa and not pi)):
                Mi[name] += int(v)
            Mi["n"] += 1
            if not pa:
                miss_by_class[yc] += 1
                rescued_by_class[yc] += int(pi)
    seeds_full = [s for s in SEEDS if sum(1 for kk, ss in keys if ss == s) == len(DAYOUT_FOLDS)]
    avg_f1 = {s: prf_from_cm(per_seed_cm[s])["macro_f1"] for s in seeds_full}
    print(f"\n    {label}: {len(keys)} (fold, seed) pairs on identical windows ✅ (counts summed over folds and seeds)")
    for lvl, D in (("window", W), ("mission", Mi)):
        n = D["n"]
        oracle = (D["both right"] + D["audio only right"] + D["IMU only right"]) / n
        print(f"      {lvl:<8} level (n = {n}): " + " | ".join(f"{k} {D[k]} ({100 * D[k] / n:.1f}%)" for k in
                                                          ("both right", "audio only right", "IMU only right", "both wrong"))
              + f" | audio alone {100 * (D['both right'] + D['audio only right']) / n:.1f}% vs oracle ceiling (either right) "
              f"{100 * oracle:.1f}%")
    print("      mission-level audio misses rescued by the IMU model, per class: " + ", ".join(
        f"C{c} {rescued_by_class[c]}/{miss_by_class[c]}" + (f" ({100 * rescued_by_class[c] / miss_by_class[c]:.0f}%)" if miss_by_class[c] else "")
        for c in range(N_CLASSES)) + "  [units: mission x seed]")
    if avg_f1:
        print(f"      ANALYSIS ONLY — NOT A PROPOSED MODEL: macro-F1 of averaging the two trained models' probabilities, pooled "
              f"day-out: " + ", ".join(f"seed {s} {v:.4f}" for s, v in avg_f1.items()) + f" | mean {np.mean(list(avg_f1.values())):.4f}"
              + (f" (vs {audio_arm} {POOLED[audio_arm]['point']['macro_f1']:.4f}, {imu_arm} {POOLED[imu_arm]['point']['macro_f1']:.4f})"
                 if audio_arm in POOLED and imu_arm in POOLED else ""))
    return {"window": dict(W), "mission": dict(Mi), "avg_f1": avg_f1, "missed": miss_by_class, "rescued": rescued_by_class}


print("\n" + "=" * 118 + "\nCOMPLEMENTARITY (analysis only, no new model) — day-out, paired on identical windows\n" + "=" * 118)
COMPLEMENT = {pair: complementarity(*pair, f"{pair[0]} (audio) x {pair[1]} (IMU)") for pair in (("A1", "S1"), ("A1", "S2"))}
if ("FUSED_v27", "day_out", 0, SEEDS[0]) in PRED:
    print("\n    Paired agreement with the FUSED v27 model (same windows):")
    for other in ("A1", "S1"):
        complementarity("FUSED_v27", other, f"FUSED_v27 x {other} (read 'audio' as FUSED)")
else:
    print(f"\n    Paired agreement with the fused model SKIPPED: {FUSED_MODE}.")

# --- 5b. standard split (Stage B) next to v24's fused Condition D ---
print("\n" + "=" * 118 + "\nSTANDARD SPLIT (Stage B, v24's original split) — macro-F1 per seed\n" + "=" * 118)
for a in ARMS:
    vals = {s: float(ROWS[(a, "standard", 0, s)]["macro_f1"]) for s in SEEDS_STANDARD if (a, "standard", 0, s) in ROWS}
    print(f"    {a}: " + ", ".join(f"seed {s} {v:.4f}" for s, v in vals.items()) + (f" | mean {np.mean(list(vals.values())):.4f}" if vals else "—"))
print("    FUSED Condition D (v24): " + ", ".join(f"seed {s} {v:.4f}" for s, v in V24_FUSED_STANDARD_F1.items())
      + f" | mean {np.mean(list(V24_FUSED_STANDARD_F1.values())):.4f}")

# --- 7. matched-random minus day-out gap per arm (only if Stage C ran) ---
if any(key[1] == "matched_random" for key in ROWS):
    print("\n" + "=" * 118 + "\nMATCHED-RANDOM minus DAY-OUT macro-F1 per arm (paired by fold, seed-averaged)\n" + "=" * 118)
    for a in ARMS:
        g = []
        for k in DAYOUT_FOLDS:
            mr = [float(ROWS[(a, "matched_random", k, s)]["macro_f1"]) for s in SEEDS if (a, "matched_random", k, s) in ROWS]
            if mr and PER_FOLD.get((a, k)):
                g.append(np.mean(mr) - PER_FOLD[(a, k)][0])
        if g:
            print(f"    {a}: gaps " + ", ".join(f"{x:+.3f}" for x in g) + f" | mean {np.mean(g):+.4f} | SD "
                  f"{np.std(g, ddof=1) if len(g) > 1 else float('nan'):.4f} | gap > 0 in {sum(x > 0 for x in g)}/{len(g)}")
else:
    print("\n[STAGE C] matched-random not run (RUN_MATCHED_RANDOM = False): no per-arm day-effect gap.")

## 11. Analysis 6 — where audio fails (descriptive, no training)
For the two class pairs A1 confuses most, Cohen's d is computed per feature, with a mission-level bootstrap CI and the
number of days on which the class difference has the same sign. The features are:
- **Audio:** mission-mean log power of the A1 representation (80 cells).
- **IMU:** 6 axes × {log std, log band power in 2–10, 10–30 and 30–90 Hz}. These use the **raw IMU samples inside each
  window's span**, because the 128-point model input is too coarse for these bands. Bands above a window's Nyquist are
  reported as NaN.

These numbers describe where the classes differ on these 95 missions. They are not performance claims.

In [ ]:
# ==============================================================================
# ANALYSIS 6 — WHERE AUDIO FAILS (descriptive; all 95 missions; no training): Cohen's d on mission-mean features for
# the class pairs A1 confuses most, with mission-level bootstrap CIs and day-sign consistency. Same for IMU descriptors.
# ==============================================================================
MISSION_NAMES = [m.name for m in ALL_MISSIONS]
MISSION_CLASS = np.array([LABEL[m] for m in ALL_MISSIONS])
MISSION_DAY = np.array([DAY[m] for m in ALL_MISSIONS])


def cohen_d(X, a_idx, b_idx):
    """Per feature: (mean_a - mean_b) / pooled SD (ddof = 1)."""
    A, B = X[a_idx], X[b_idx]
    na, nb = len(A), len(B)
    sp = np.sqrt(((na - 1) * np.nanvar(A, axis=0, ddof=1) + (nb - 1) * np.nanvar(B, axis=0, ddof=1)) / (na + nb - 2))
    return (np.nanmean(A, axis=0) - np.nanmean(B, axis=0)) / np.where(sp > 0, sp, np.nan)


def describe_pair(X, names, ci, cj, label):
    a_idx, b_idx = np.where(MISSION_CLASS == ci)[0], np.where(MISSION_CLASS == cj)[0]
    d = cohen_d(X, a_idx, b_idx)
    rng = np.random.default_rng(BOOTSTRAP_SEED)
    boot = np.array([cohen_d(X, rng.choice(a_idx, len(a_idx)), rng.choice(b_idx, len(b_idx))) for _ in range(BOOTSTRAP_RESAMPLES)])
    lo, hi = np.nanpercentile(boot, 2.5, axis=0), np.nanpercentile(boot, 97.5, axis=0)
    same, n_days = np.zeros(X.shape[1], int), 0
    for day in DAYS:
        ai, bi = a_idx[MISSION_DAY[a_idx] == day], b_idx[MISSION_DAY[b_idx] == day]
        if len(ai) and len(bi):
            n_days += 1
            same += (np.sign(np.nanmean(X[ai], axis=0) - np.nanmean(X[bi], axis=0)) == np.sign(d)).astype(int)
    order = np.argsort(-np.nan_to_num(np.abs(d), nan=-1))[:TOP_CELLS]
    print(f"\n    {label}: class {ci} ({len(a_idx)} missions) vs class {cj} ({len(b_idx)} missions) | top {TOP_CELLS} by |d| "
          f"(✓ = same sign on >= {SIGN_CONSISTENT_MIN_DAYS} of {n_days} days)")
    print(f"      {'feature':<26}{'d':>8}{'95% CI (missions)':>22}{'days same sign':>16}")
    for i in order:
        print(f"      {names[i]:<26}{d[i]:>+8.2f}{f'[{lo[i]:+.2f}, {hi[i]:+.2f}]':>22}{f'{same[i]}/{n_days}':>14}"
              f"{'  ✓' if same[i] >= SIGN_CONSISTENT_MIN_DAYS else ''}")
    return {"d": d, "lo": lo, "hi": hi, "same_days": same, "n_days": n_days}


# --- the two class pairs A1 confuses most (pooled day-out, summed over seeds and folds) ---
WHERE = {}
if "A1" in POOLED:
    cm = POOLED["A1"]["cm_sum"]
    rates = {(i, j): (cm[i, j] + cm[j, i]) / (cm[i].sum() + cm[j].sum()) for i in range(N_CLASSES) for j in range(i + 1, N_CLASSES)}
    CONFUSED_PAIRS = sorted(rates, key=lambda p: -rates[p])[:N_CONFUSED_PAIRS]
    print("=" * 118 + "\nWHERE AUDIO FAILS — descriptive (all 95 missions, no training): where the classes differ, NOT a performance claim\n" + "=" * 118)
    print("    A1 pooled day-out confusion rate per class pair (both directions / both classes' windows), top 5: " + ", ".join(
        f"C{i}<->C{j} {rates[(i, j)]:.3f}" for i, j in sorted(rates, key=lambda p: -rates[p])[:5]))
    print(f"    -> analysing {', '.join(f'C{i} vs C{j}' for i, j in CONFUSED_PAIRS)}")

    # audio: mission-mean log power of the A1 representation (5 channels x bins 1-16)
    XA = np.stack([A1_LOGPOW[n] for n in MISSION_NAMES])
    names_a = [f"audio ch{c + 1} bin{b:>2}" for c in range(A1_KEEP_SLOTS) for b in A1_BINS]
    print("\n  AUDIO — mission-mean log power, A1 representation (5 channels x 16 bins = 80 cells)")
    for ci, cj in CONFUSED_PAIRS:
        WHERE[("audio", ci, cj)] = describe_pair(XA, names_a, ci, cj, f"C{ci} vs C{cj}")

    # IMU: per window, from the RAW IMU samples inside the window span (the 128-point model input is too coarse for these bands)
    def _window_starts(raw):
        s_times, a_times = raw["sensor_times"], raw["audio_times"]
        max_time, starts = min(s_times[-1], a_times[-1]), []
        t_start = max(0.0, float(a_times[0])) if CORRECTED_WINDOW_START == "audio_onset" else 0.0
        while t_start + WINDOW_SEC <= max_time:
            if (((s_times >= t_start) & (s_times <= t_start + WINDOW_SEC)).sum()) >= MIN_SENSOR_SAMPLES_PER_WINDOW:
                starts.append(t_start)
            t_start += STEP_SEC
        return starts

    AXES = ["acc_x", "acc_y", "acc_z", "gyr_x", "gyr_y", "gyr_z"]
    names_i = [f"IMU {ax} log std" for ax in AXES] + [f"IMU {ax} log P {lo}-{hi}Hz" for (lo, hi) in IMU_BANDS_HZ for ax in AXES]
    XI, FS = [], []
    for m in tqdm(ALL_MISSIONS, desc="IMU descriptors", leave=False):
        raw = CORRECTED_PARSED[m]
        s_times, tel = raw["sensor_times"], raw["telemetry"].astype(np.float64)
        rows = []
        for t in _window_starts(raw):
            x = tel[(s_times >= t) & (s_times <= t + WINDOW_SEC)]            # samples in FILE order, as the pipeline selects them
            fs = len(x) / WINDOW_SEC
            FS.append(fs)
            xc = x - x.mean(axis=0)
            spec = np.abs(np.fft.rfft(xc, axis=0)) ** 2 / len(x) ** 2
            freqs = np.fft.rfftfreq(len(x), d=1.0 / fs)
            feats = list(np.log(x.std(axis=0) + 1e-12))
            for lo, hi in IMU_BANDS_HZ:
                band = (freqs >= lo) & (freqs < hi)
                feats += list(np.log(spec[band].sum(axis=0) + 1e-12)) if lo < fs / 2 and band.any() else [np.nan] * N_SENSOR_CHANNELS
            rows.append(feats)
        XI.append(np.nanmean(np.array(rows, dtype=float), axis=0))
    XI, FS = np.array(XI), np.array(FS)
    print(f"\n  IMU — raw samples per window give an effective rate of median {np.median(FS):.0f} Hz (min {FS.min():.0f}, max "
          f"{FS.max():.0f}) -> Nyquist ~{np.median(FS) / 2:.0f} Hz. Bands above a window's Nyquist are NaN; a band cut by it "
          f"is truncated. The IMU timestamps are not monotonic in every mission (v24), so the rate is count / {WINDOW_SEC} s and "
          f"samples are taken in file order, as the pipeline does.")
    for (lo, hi) in IMU_BANDS_HZ:
        print(f"    band {lo}-{hi} Hz: " + ("fully below Nyquist" if hi <= np.median(FS) / 2 else
                                           ("TRUNCATED at Nyquist" if lo < np.median(FS) / 2 else "ABOVE Nyquist: NaN")))
    for ci, cj in CONFUSED_PAIRS:
        WHERE[("imu", ci, cj)] = describe_pair(XI, names_i, ci, cj, f"C{ci} vs C{cj}")
else:
    print("⚠️ WHERE AUDIO FAILS skipped: A1 has no seed with all 5 day-out folds complete.")

print("\n" + "!" * 118)
print("!  SCOPE: n = 95 missions, about 19 per class, so a per-class recall is only known to roughly ±0.1 (see the mission-level")
print(f"!  CIs). {len(SEEDS)} seeds per fold. This notebook CHARACTERIZES each modality; it is not a hypothesis test and not a")
print("!  modality contest. Cohen's d values describe where classes differ on these 95 missions, not what a model can learn.")
print("!" * 118)

## 12. Final validation

In [ ]:
# ==============================================================================
# FINAL VALIDATION — every verification hook in one place
# ==============================================================================
_uni = [r for r in load_rows() if r["arm"] in ARMS]
print("=" * 110)
print("FINAL VALIDATION")
print("=" * 110)
checks = {
    "Stage 0 gate (a): original split data == v24/v27 (incl. window sha1)": GATE_A_PASS,
    "Stage 0 gate (b): fused D seed 42 reproduces v24 (±0.0005)": GATE_B_PASS,
    "shared windowing reproduces v24's windows (standard split, A2 + sensor)": WINDOWING_MATCHES_V24,
    "A1: slot 6 all-zero in 100% of buffers": _n_zero6 == _n_buf,
    "window ids identical across arms (every fold/protocol built this session)": all(
        all(v == d[ARMS[0]] for v in d.values()) for d in WINDOW_ID_SHA1.values()),
    "S2: mean |window mean| after removal ~0 (< 1e-4)": all(v < 1e-4 for v in S2_CHECK.values()),
    "independence + CE-only loss asserted for every unimodal run": True,       # check_independence runs before every training
    f"all {len(PLANNED)} planned unimodal runs in the CSV": not MISSING,
    "a probability file for every unimodal row": all(Path(PROBS_DIR / f"{r['arm']}_{r['protocol']}_{r['fold']}_{r['seed']}.npz").exists()
                                                     for r in _uni),
    "fusion backbone is real mamba_ssm": MAMBA_CLASS is not None,
}
for name, ok in checks.items():
    print(f"  {name:<74}: {'✅ pass' if ok else '🚨 FAIL'}")
if not S2_CHECK:
    print(f"  {'(S2 check not run this session: every S2 run was already in the CSV)':<74}")
_secs = defaultdict(list)
for r in _uni:
    _secs[r["arm"]].append(float(r["train_seconds"]))
print(f"  {'training time per run, by arm':<74}: " + ", ".join(f"{a} {np.mean(v) / 60:.1f} min" for a, v in _secs.items()))
print(f"  {'fused reference':<74}: {FUSED_MODE}")
print(f"  {'Stage C (matched-random)':<74}: {'ran' if RUN_MATCHED_RANDOM else 'OFF (RUN_MATCHED_RANDOM = False)'}")
print(f"  {'outputs':<74}: {RESULTS_CSV.name}, {PROBS_DIR.name}/*.npz, {CHECKPOINT_DIR.name}/")
print("=" * 110)
if not all(checks.values()):
    print("🚨 at least one check FAILED — read the flagged line before using the characterization.")